# Argmin with one packed 64-bit atomic

An arg-reduction normally carries two values (the best value and where it is). Atomics work on **one** word, but you can make a pair fit in one word: put the value in the high 32 bits and the index in the low 32 bits,

$$\text{key}(v, i) = v \cdot 2^{32} + i, \qquad 0 \le i < 2^{32}$$

Comparing two keys as signed 64-bit integers compares the values first and breaks ties by the smaller index. So `cuda.atomic.min` on keys computes the argmin, ties going to the first occurrence, in a single kernel.

Write `argmin(x)` for a 1-D `int32` array.

## Requirements
- Fixed grid: **4 blocks of 64 threads**, grid-stride loop.
- Each thread computes the smallest key of its own elements in a local `int64`, then issues **one** `cuda.atomic.min` into a 1-element `int64` device array initialised to the largest `int64` ($2^{63} - 1$). Threads with no elements skip the atomic.
- Build the key as `(int64(v) << 32) + i`. Every `int32` value fits: $v \cdot 2^{32}$ lies in $[-2^{63}, 2^{63} - 2^{32}]$, and adding $i < 2^{32}$ cannot overflow.
- Decode the index on the host from the low 32 bits: `key & 0xFFFFFFFF`.

## Inputs and output
- `x`: `int32`, shape `(n,)`, `1 <= n <= 4096`; any `int32` values, including the extremes.
- Return the index as a Python `int`, equal to `np.argmin(x)`.

Integer min is exact and order-independent, so the result is deterministic.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda, int64

BLOCKS, TPB = 4, 64
KEY_MAX = np.iinfo(np.int64).max

def argmin(x):
    """x: int32 (n,), n >= 1. Returns the index of the first minimum as a Python int."""
    d_x = cuda.to_device(x)
    d_best = cuda.to_device(np.full(1, KEY_MAX, dtype=np.int64))
    argmin_kernel[BLOCKS, TPB](d_x, d_best)
    return int(d_best.copy_to_host()[0])   # TODO: this is the key, not the index

@cuda.jit
def argmin_kernel(x, best):
    i = cuda.grid(1)
    if i < x.shape[0]:
        cuda.atomic.min(best, 0, x[i])     # TODO: pack (value, index) into one int64 key


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [5, 2, 8, 2, 9], "dtype": "int32"}], "e": 1}, {"a": [{"__nd__": [-3, 4, -10, 0], "dtype": "int32"}], "e": 2}, {"a": [{"__nd__": [17], "dtype": "int32"}], "e": 0}, {"a": [{"__nd__": [2147483647, -2147483648, -2147483648, 0], "dtype": "int32"}], "e": 1}, {"a": [{"__nd__": [2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647, 2147483647], "dtype": "int32"}], "e": 0}, {"a": [{"__nd__": [4000, 3999, 3998, 3997, 3996, 3995, 3994, 3993, 3992, 3991, 3990, 3989, 3988, 3987, 3986, 3985, 3984, 3983, 3982, 3981, 3980, 3979, 3978, 3977, 3976, 3975, 3974, 3973, 3972, 3971, 3970, 3969, 3968, 3967, 3966, 3965, 3964, 3963, 3962, 3961, 3960, 3959, 3958, 3957, 3956, 3955, 3954, 3953, 3952, 3951, 3950, 3949, 3948, 3947, 3946, 3945, 3944, 3943, 3942, 3941, 3940, 3939, 3938, 3937, 3936, 3935, 3934, 3933, 3932, 3931, 3930, 3929, 3928, 3927, 3926, 3925, 3924, 3923, 3922, 3921, 3920, 3919, 3918, 3917, 3916, 3915, 3914, 3913, 3912, 3911, 3910, 3909, 3908, 3907, 3906, 3905, 3904, 3903, 3902, 3901, 3900, 3899, 3898, 3897, 3896, 3895, 3894, 3893, 3892, 3891, 3890, 3889, 3888, 3887, 3886, 3885, 3884, 3883, 3882, 3881, 3880, 3879, 3878, 3877, 3876, 3875, 3874, 3873, 3872, 3871, 3870, 3869, 3868, 3867, 3866, 3865, 3864, 3863, 3862, 3861, 3860, 3859, 3858, 3857, 3856, 3855, 3854, 3853, 3852, 3851, 3850, 3849, 3848, 3847, 3846, 3845, 3844, 3843, 3842, 3841, 3840, 3839, 3838, 3837, 3836, 3835, 3834, 3833, 3832, 3831, 3830, 3829, 3828, 3827, 3826, 3825, 3824, 3823, 3822, 3821, 3820, 3819, 3818, 3817, 3816, 3815, 3814, 3813, 3812, 3811, 3810, 3809, 3808, 3807, 3806, 3805, 3804, 3803, 3802, 3801, 3800, 3799, 3798, 3797, 3796, 3795, 3794, 3793, 3792, 3791, 3790, 3789, 3788, 3787, 3786, 3785, 3784, 3783, 3782, 3781, 3780, 3779, 3778, 3777, 3776, 3775, 3774, 3773, 3772, 3771, 3770, 3769, 3768, 3767, 3766, 3765, 3764, 3763, 3762, 3761, 3760, 3759, 3758, 3757, 3756, 3755, 3754, 3753, 3752, 3751, 3750, 3749, 3748, 3747, 3746, 3745, 3744, 3743, 3742, 3741, 3740, 3739, 3738, 3737, 3736, 3735, 3734, 3733, 3732, 3731, 3730, 3729, 3728, 3727, 3726, 3725, 3724, 3723, 3722, 3721, 3720, 3719, 3718, 3717, 3716, 3715, 3714, 3713, 3712, 3711, 3710, 3709, 3708, 3707, 3706, 3705, 3704, 3703, 3702, 3701, 3700, 3699, 3698, 3697, 3696, 3695, 3694, 3693, 3692, 3691, 3690, 3689, 3688, 3687, 3686, 3685, 3684, 3683, 3682, 3681, 3680, 3679, 3678, 3677, 3676, 3675, 3674, 3673, 3672, 3671, 3670, 3669, 3668, 3667, 3666, 3665, 3664, 3663, 3662, 3661, 3660, 3659, 3658, 3657, 3656, 3655, 3654, 3653, 3652, 3651, 3650, 3649, 3648, 3647, 3646, 3645, 3644, 3643, 3642, 3641, 3640, 3639, 3638, 3637, 3636, 3635, 3634, 3633, 3632, 3631, 3630, 3629, 3628, 3627, 3626, 3625, 3624, 3623, 3622, 3621, 3620, 3619, 3618, 3617, 3616, 3615, 3614, 3613, 3612, 3611, 3610, 3609, 3608, 3607, 3606, 3605, 3604, 3603, 3602, 3601, 3600, 3599, 3598, 3597, 3596, 3595, 3594, 3593, 3592, 3591, 3590, 3589, 3588, 3587, 3586, 3585, 3584, 3583, 3582, 3581, 3580, 3579, 3578, 3577, 3576, 3575, 3574, 3573, 3572, 3571, 3570, 3569, 3568, 3567, 3566, 3565, 3564, 3563, 3562, 3561, 3560, 3559, 3558, 3557, 3556, 3555, 3554, 3553, 3552, 3551, 3550, 3549, 3548, 3547, 3546, 3545, 3544, 3543, 3542, 3541, 3540, 3539, 3538, 3537, 3536, 3535, 3534, 3533, 3532, 3531, 3530, 3529, 3528, 3527, 3526, 3525, 3524, 3523, 3522, 3521, 3520, 3519, 3518, 3517, 3516, 3515, 3514, 3513, 3512, 3511, 3510, 3509, 3508, 3507, 3506, 3505, 3504, 3503, 3502, 3501, 3500, 3499, 3498, 3497, 3496, 3495, 3494, 3493, 3492, 3491, 3490, 3489, 3488, 3487, 3486, 3485, 3484, 3483, 3482, 3481, 3480, 3479, 3478, 3477, 3476, 3475, 3474, 3473, 3472, 3471, 3470, 3469, 3468, 3467, 3466, 3465, 3464, 3463, 3462, 3461, 3460, 3459, 3458, 3457, 3456, 3455, 3454, 3453, 3452, 3451, 3450, 3449, 3448, 3447, 3446, 3445, 3444, 3443, 3442, 3441, 3440, 3439, 3438, 3437, 3436, 3435, 3434, 3433, 3432, 3431, 3430, 3429, 3428, 3427, 3426, 3425, 3424, 3423, 3422, 3421, 3420, 3419, 3418, 3417, 3416, 3415, 3414, 3413, 3412, 3411, 3410, 3409, 3408, 3407, 3406, 3405, 3404, 3403, 3402, 3401, 3400, 3399, 3398, 3397, 3396, 3395, 3394, 3393, 3392, 3391, 3390, 3389, 3388, 3387, 3386, 3385, 3384, 3383, 3382, 3381, 3380, 3379, 3378, 3377, 3376, 3375, 3374, 3373, 3372, 3371, 3370, 3369, 3368, 3367, 3366, 3365, 3364, 3363, 3362, 3361, 3360, 3359, 3358, 3357, 3356, 3355, 3354, 3353, 3352, 3351, 3350, 3349, 3348, 3347, 3346, 3345, 3344, 3343, 3342, 3341, 3340, 3339, 3338, 3337, 3336, 3335, 3334, 3333, 3332, 3331, 3330, 3329, 3328, 3327, 3326, 3325, 3324, 3323, 3322, 3321, 3320, 3319, 3318, 3317, 3316, 3315, 3314, 3313, 3312, 3311, 3310, 3309, 3308, 3307, 3306, 3305, 3304, 3303, 3302, 3301, 3300, 3299, 3298, 3297, 3296, 3295, 3294, 3293, 3292, 3291, 3290, 3289, 3288, 3287, 3286, 3285, 3284, 3283, 3282, 3281, 3280, 3279, 3278, 3277, 3276, 3275, 3274, 3273, 3272, 3271, 3270, 3269, 3268, 3267, 3266, 3265, 3264, 3263, 3262, 3261, 3260, 3259, 3258, 3257, 3256, 3255, 3254, 3253, 3252, 3251, 3250, 3249, 3248, 3247, 3246, 3245, 3244, 3243, 3242, 3241, 3240, 3239, 3238, 3237, 3236, 3235, 3234, 3233, 3232, 3231, 3230, 3229, 3228, 3227, 3226, 3225, 3224, 3223, 3222, 3221, 3220, 3219, 3218, 3217, 3216, 3215, 3214, 3213, 3212, 3211, 3210, 3209, 3208, 3207, 3206, 3205, 3204, 3203, 3202, 3201, 3200, 3199, 3198, 3197, 3196, 3195, 3194, 3193, 3192, 3191, 3190, 3189, 3188, 3187, 3186, 3185, 3184, 3183, 3182, 3181, 3180, 3179, 3178, 3177, 3176, 3175, 3174, 3173, 3172, 3171, 3170, 3169, 3168, 3167, 3166, 3165, 3164, 3163, 3162, 3161, 3160, 3159, 3158, 3157, 3156, 3155, 3154, 3153, 3152, 3151, 3150, 3149, 3148, 3147, 3146, 3145, 3144, 3143, 3142, 3141, 3140, 3139, 3138, 3137, 3136, 3135, 3134, 3133, 3132, 3131, 3130, 3129, 3128, 3127, 3126, 3125, 3124, 3123, 3122, 3121, 3120, 3119, 3118, 3117, 3116, 3115, 3114, 3113, 3112, 3111, 3110, 3109, 3108, 3107, 3106, 3105, 3104, 3103, 3102, 3101, 3100, 3099, 3098, 3097, 3096, 3095, 3094, 3093, 3092, 3091, 3090, 3089, 3088, 3087, 3086, 3085, 3084, 3083, 3082, 3081, 3080, 3079, 3078, 3077, 3076, 3075, 3074, 3073, 3072, 3071, 3070, 3069, 3068, 3067, 3066, 3065, 3064, 3063, 3062, 3061, 3060, 3059, 3058, 3057, 3056, 3055, 3054, 3053, 3052, 3051, 3050, 3049, 3048, 3047, 3046, 3045, 3044, 3043, 3042, 3041, 3040, 3039, 3038, 3037, 3036, 3035, 3034, 3033, 3032, 3031, 3030, 3029, 3028, 3027, 3026, 3025, 3024, 3023, 3022, 3021, 3020, 3019, 3018, 3017, 3016, 3015, 3014, 3013, 3012, 3011, 3010, 3009, 3008, 3007, 3006, 3005, 3004, 3003, 3002, 3001, 3000, 2999, 2998, 2997, 2996, 2995, 2994, 2993, 2992, 2991, 2990, 2989, 2988, 2987, 2986, 2985, 2984, 2983, 2982, 2981, 2980, 2979, 2978, 2977, 2976, 2975, 2974, 2973, 2972, 2971, 2970, 2969, 2968, 2967, 2966, 2965, 2964, 2963, 2962, 2961, 2960, 2959, 2958, 2957, 2956, 2955, 2954, 2953, 2952, 2951, 2950, 2949, 2948, 2947, 2946, 2945, 2944, 2943, 2942, 2941, 2940, 2939, 2938, 2937, 2936, 2935, 2934, 2933, 2932, 2931, 2930, 2929, 2928, 2927, 2926, 2925, 2924, 2923, 2922, 2921, 2920, 2919, 2918, 2917, 2916, 2915, 2914, 2913, 2912, 2911, 2910, 2909, 2908, 2907, 2906, 2905, 2904, 2903, 2902, 2901, 2900, 2899, 2898, 2897, 2896, 2895, 2894, 2893, 2892, 2891, 2890, 2889, 2888, 2887, 2886, 2885, 2884, 2883, 2882, 2881, 2880, 2879, 2878, 2877, 2876, 2875, 2874, 2873, 2872, 2871, 2870, 2869, 2868, 2867, 2866, 2865, 2864, 2863, 2862, 2861, 2860, 2859, 2858, 2857, 2856, 2855, 2854, 2853, 2852, 2851, 2850, 2849, 2848, 2847, 2846, 2845, 2844, 2843, 2842, 2841, 2840, 2839, 2838, 2837, 2836, 2835, 2834, 2833, 2832, 2831, 2830, 2829, 2828, 2827, 2826, 2825, 2824, 2823, 2822, 2821, 2820, 2819, 2818, 2817, 2816, 2815, 2814, 2813, 2812, 2811, 2810, 2809, 2808, 2807, 2806, 2805, 2804, 2803, 2802, 2801, 2800, 2799, 2798, 2797, 2796, 2795, 2794, 2793, 2792, 2791, 2790, 2789, 2788, 2787, 2786, 2785, 2784, 2783, 2782, 2781, 2780, 2779, 2778, 2777, 2776, 2775, 2774, 2773, 2772, 2771, 2770, 2769, 2768, 2767, 2766, 2765, 2764, 2763, 2762, 2761, 2760, 2759, 2758, 2757, 2756, 2755, 2754, 2753, 2752, 2751, 2750, 2749, 2748, 2747, 2746, 2745, 2744, 2743, 2742, 2741, 2740, 2739, 2738, 2737, 2736, 2735, 2734, 2733, 2732, 2731, 2730, 2729, 2728, 2727, 2726, 2725, 2724, 2723, 2722, 2721, 2720, 2719, 2718, 2717, 2716, 2715, 2714, 2713, 2712, 2711, 2710, 2709, 2708, 2707, 2706, 2705, 2704, 2703, 2702, 2701, 2700, 2699, 2698, 2697, 2696, 2695, 2694, 2693, 2692, 2691, 2690, 2689, 2688, 2687, 2686, 2685, 2684, 2683, 2682, 2681, 2680, 2679, 2678, 2677, 2676, 2675, 2674, 2673, 2672, 2671, 2670, 2669, 2668, 2667, 2666, 2665, 2664, 2663, 2662, 2661, 2660, 2659, 2658, 2657, 2656, 2655, 2654, 2653, 2652, 2651, 2650, 2649, 2648, 2647, 2646, 2645, 2644, 2643, 2642, 2641, 2640, 2639, 2638, 2637, 2636, 2635, 2634, 2633, 2632, 2631, 2630, 2629, 2628, 2627, 2626, 2625, 2624, 2623, 2622, 2621, 2620, 2619, 2618, 2617, 2616, 2615, 2614, 2613, 2612, 2611, 2610, 2609, 2608, 2607, 2606, 2605, 2604, 2603, 2602, 2601, 2600, 2599, 2598, 2597, 2596, 2595, 2594, 2593, 2592, 2591, 2590, 2589, 2588, 2587, 2586, 2585, 2584, 2583, 2582, 2581, 2580, 2579, 2578, 2577, 2576, 2575, 2574, 2573, 2572, 2571, 2570, 2569, 2568, 2567, 2566, 2565, 2564, 2563, 2562, 2561, 2560, 2559, 2558, 2557, 2556, 2555, 2554, 2553, 2552, 2551, 2550, 2549, 2548, 2547, 2546, 2545, 2544, 2543, 2542, 2541, 2540, 2539, 2538, 2537, 2536, 2535, 2534, 2533, 2532, 2531, 2530, 2529, 2528, 2527, 2526, 2525, 2524, 2523, 2522, 2521, 2520, 2519, 2518, 2517, 2516, 2515, 2514, 2513, 2512, 2511, 2510, 2509, 2508, 2507, 2506, 2505, 2504, 2503, 2502, 2501, 2500, 2499, 2498, 2497, 2496, 2495, 2494, 2493, 2492, 2491, 2490, 2489, 2488, 2487, 2486, 2485, 2484, 2483, 2482, 2481, 2480, 2479, 2478, 2477, 2476, 2475, 2474, 2473, 2472, 2471, 2470, 2469, 2468, 2467, 2466, 2465, 2464, 2463, 2462, 2461, 2460, 2459, 2458, 2457, 2456, 2455, 2454, 2453, 2452, 2451, 2450, 2449, 2448, 2447, 2446, 2445, 2444, 2443, 2442, 2441, 2440, 2439, 2438, 2437, 2436, 2435, 2434, 2433, 2432, 2431, 2430, 2429, 2428, 2427, 2426, 2425, 2424, 2423, 2422, 2421, 2420, 2419, 2418, 2417, 2416, 2415, 2414, 2413, 2412, 2411, 2410, 2409, 2408, 2407, 2406, 2405, 2404, 2403, 2402, 2401, 2400, 2399, 2398, 2397, 2396, 2395, 2394, 2393, 2392, 2391, 2390, 2389, 2388, 2387, 2386, 2385, 2384, 2383, 2382, 2381, 2380, 2379, 2378, 2377, 2376, 2375, 2374, 2373, 2372, 2371, 2370, 2369, 2368, 2367, 2366, 2365, 2364, 2363, 2362, 2361, 2360, 2359, 2358, 2357, 2356, 2355, 2354, 2353, 2352, 2351, 2350, 2349, 2348, 2347, 2346, 2345, 2344, 2343, 2342, 2341, 2340, 2339, 2338, 2337, 2336, 2335, 2334, 2333, 2332, 2331, 2330, 2329, 2328, 2327, 2326, 2325, 2324, 2323, 2322, 2321, 2320, 2319, 2318, 2317, 2316, 2315, 2314, 2313, 2312, 2311, 2310, 2309, 2308, 2307, 2306, 2305, 2304, 2303, 2302, 2301, 2300, 2299, 2298, 2297, 2296, 2295, 2294, 2293, 2292, 2291, 2290, 2289, 2288, 2287, 2286, 2285, 2284, 2283, 2282, 2281, 2280, 2279, 2278, 2277, 2276, 2275, 2274, 2273, 2272, 2271, 2270, 2269, 2268, 2267, 2266, 2265, 2264, 2263, 2262, 2261, 2260, 2259, 2258, 2257, 2256, 2255, 2254, 2253, 2252, 2251, 2250, 2249, 2248, 2247, 2246, 2245, 2244, 2243, 2242, 2241, 2240, 2239, 2238, 2237, 2236, 2235, 2234, 2233, 2232, 2231, 2230, 2229, 2228, 2227, 2226, 2225, 2224, 2223, 2222, 2221, 2220, 2219, 2218, 2217, 2216, 2215, 2214, 2213, 2212, 2211, 2210, 2209, 2208, 2207, 2206, 2205, 2204, 2203, 2202, 2201, 2200, 2199, 2198, 2197, 2196, 2195, 2194, 2193, 2192, 2191, 2190, 2189, 2188, 2187, 2186, 2185, 2184, 2183, 2182, 2181, 2180, 2179, 2178, 2177, 2176, 2175, 2174, 2173, 2172, 2171, 2170, 2169, 2168, 2167, 2166, 2165, 2164, 2163, 2162, 2161, 2160, 2159, 2158, 2157, 2156, 2155, 2154, 2153, 2152, 2151, 2150, 2149, 2148, 2147, 2146, 2145, 2144, 2143, 2142, 2141, 2140, 2139, 2138, 2137, 2136, 2135, 2134, 2133, 2132, 2131, 2130, 2129, 2128, 2127, 2126, 2125, 2124, 2123, 2122, 2121, 2120, 2119, 2118, 2117, 2116, 2115, 2114, 2113, 2112, 2111, 2110, 2109, 2108, 2107, 2106, 2105, 2104, 2103, 2102, 2101, 2100, 2099, 2098, 2097, 2096, 2095, 2094, 2093, 2092, 2091, 2090, 2089, 2088, 2087, 2086, 2085, 2084, 2083, 2082, 2081, 2080, 2079, 2078, 2077, 2076, 2075, 2074, 2073, 2072, 2071, 2070, 2069, 2068, 2067, 2066, 2065, 2064, 2063, 2062, 2061, 2060, 2059, 2058, 2057, 2056, 2055, 2054, 2053, 2052, 2051, 2050, 2049, 2048, 2047, 2046, 2045, 2044, 2043, 2042, 2041, 2040, 2039, 2038, 2037, 2036, 2035, 2034, 2033, 2032, 2031, 2030, 2029, 2028, 2027, 2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008, 2007, 2006, 2005, 2004, 2003, 2002, 2001, 2000, 1999, 1998, 1997, 1996, 1995, 1994, 1993, 1992, 1991, 1990, 1989, 1988, 1987, 1986, 1985, 1984, 1983, 1982, 1981, 1980, 1979, 1978, 1977, 1976, 1975, 1974, 1973, 1972, 1971, 1970, 1969, 1968, 1967, 1966, 1965, 1964, 1963, 1962, 1961, 1960, 1959, 1958, 1957, 1956, 1955, 1954, 1953, 1952, 1951, 1950, 1949, 1948, 1947, 1946, 1945, 1944, 1943, 1942, 1941, 1940, 1939, 1938, 1937, 1936, 1935, 1934, 1933, 1932, 1931, 1930, 1929, 1928, 1927, 1926, 1925, 1924, 1923, 1922, 1921, 1920, 1919, 1918, 1917, 1916, 1915, 1914, 1913, 1912, 1911, 1910, 1909, 1908, 1907, 1906, 1905, 1904, 1903, 1902, 1901, 1900, 1899, 1898, 1897, 1896, 1895, 1894, 1893, 1892, 1891, 1890, 1889, 1888, 1887, 1886, 1885, 1884, 1883, 1882, 1881, 1880, 1879, 1878, 1877, 1876, 1875, 1874, 1873, 1872, 1871, 1870, 1869, 1868, 1867, 1866, 1865, 1864, 1863, 1862, 1861, 1860, 1859, 1858, 1857, 1856, 1855, 1854, 1853, 1852, 1851, 1850, 1849, 1848, 1847, 1846, 1845, 1844, 1843, 1842, 1841, 1840, 1839, 1838, 1837, 1836, 1835, 1834, 1833, 1832, 1831, 1830, 1829, 1828, 1827, 1826, 1825, 1824, 1823, 1822, 1821, 1820, 1819, 1818, 1817, 1816, 1815, 1814, 1813, 1812, 1811, 1810, 1809, 1808, 1807, 1806, 1805, 1804, 1803, 1802, 1801, 1800, 1799, 1798, 1797, 1796, 1795, 1794, 1793, 1792, 1791, 1790, 1789, 1788, 1787, 1786, 1785, 1784, 1783, 1782, 1781, 1780, 1779, 1778, 1777, 1776, 1775, 1774, 1773, 1772, 1771, 1770, 1769, 1768, 1767, 1766, 1765, 1764, 1763, 1762, 1761, 1760, 1759, 1758, 1757, 1756, 1755, 1754, 1753, 1752, 1751, 1750, 1749, 1748, 1747, 1746, 1745, 1744, 1743, 1742, 1741, 1740, 1739, 1738, 1737, 1736, 1735, 1734, 1733, 1732, 1731, 1730, 1729, 1728, 1727, 1726, 1725, 1724, 1723, 1722, 1721, 1720, 1719, 1718, 1717, 1716, 1715, 1714, 1713, 1712, 1711, 1710, 1709, 1708, 1707, 1706, 1705, 1704, 1703, 1702, 1701, 1700, 1699, 1698, 1697, 1696, 1695, 1694, 1693, 1692, 1691, 1690, 1689, 1688, 1687, 1686, 1685, 1684, 1683, 1682, 1681, 1680, 1679, 1678, 1677, 1676, 1675, 1674, 1673, 1672, 1671, 1670, 1669, 1668, 1667, 1666, 1665, 1664, 1663, 1662, 1661, 1660, 1659, 1658, 1657, 1656, 1655, 1654, 1653, 1652, 1651, 1650, 1649, 1648, 1647, 1646, 1645, 1644, 1643, 1642, 1641, 1640, 1639, 1638, 1637, 1636, 1635, 1634, 1633, 1632, 1631, 1630, 1629, 1628, 1627, 1626, 1625, 1624, 1623, 1622, 1621, 1620, 1619, 1618, 1617, 1616, 1615, 1614, 1613, 1612, 1611, 1610, 1609, 1608, 1607, 1606, 1605, 1604, 1603, 1602, 1601, 1600, 1599, 1598, 1597, 1596, 1595, 1594, 1593, 1592, 1591, 1590, 1589, 1588, 1587, 1586, 1585, 1584, 1583, 1582, 1581, 1580, 1579, 1578, 1577, 1576, 1575, 1574, 1573, 1572, 1571, 1570, 1569, 1568, 1567, 1566, 1565, 1564, 1563, 1562, 1561, 1560, 1559, 1558, 1557, 1556, 1555, 1554, 1553, 1552, 1551, 1550, 1549, 1548, 1547, 1546, 1545, 1544, 1543, 1542, 1541, 1540, 1539, 1538, 1537, 1536, 1535, 1534, 1533, 1532, 1531, 1530, 1529, 1528, 1527, 1526, 1525, 1524, 1523, 1522, 1521, 1520, 1519, 1518, 1517, 1516, 1515, 1514, 1513, 1512, 1511, 1510, 1509, 1508, 1507, 1506, 1505, 1504, 1503, 1502, 1501, 1500, 1499, 1498, 1497, 1496, 1495, 1494, 1493, 1492, 1491, 1490, 1489, 1488, 1487, 1486, 1485, 1484, 1483, 1482, 1481, 1480, 1479, 1478, 1477, 1476, 1475, 1474, 1473, 1472, 1471, 1470, 1469, 1468, 1467, 1466, 1465, 1464, 1463, 1462, 1461, 1460, 1459, 1458, 1457, 1456, 1455, 1454, 1453, 1452, 1451, 1450, 1449, 1448, 1447, 1446, 1445, 1444, 1443, 1442, 1441, 1440, 1439, 1438, 1437, 1436, 1435, 1434, 1433, 1432, 1431, 1430, 1429, 1428, 1427, 1426, 1425, 1424, 1423, 1422, 1421, 1420, 1419, 1418, 1417, 1416, 1415, 1414, 1413, 1412, 1411, 1410, 1409, 1408, 1407, 1406, 1405, 1404, 1403, 1402, 1401, 1400, 1399, 1398, 1397, 1396, 1395, 1394, 1393, 1392, 1391, 1390, 1389, 1388, 1387, 1386, 1385, 1384, 1383, 1382, 1381, 1380, 1379, 1378, 1377, 1376, 1375, 1374, 1373, 1372, 1371, 1370, 1369, 1368, 1367, 1366, 1365, 1364, 1363, 1362, 1361, 1360, 1359, 1358, 1357, 1356, 1355, 1354, 1353, 1352, 1351, 1350, 1349, 1348, 1347, 1346, 1345, 1344, 1343, 1342, 1341, 1340, 1339, 1338, 1337, 1336, 1335, 1334, 1333, 1332, 1331, 1330, 1329, 1328, 1327, 1326, 1325, 1324, 1323, 1322, 1321, 1320, 1319, 1318, 1317, 1316, 1315, 1314, 1313, 1312, 1311, 1310, 1309, 1308, 1307, 1306, 1305, 1304, 1303, 1302, 1301, 1300, 1299, 1298, 1297, 1296, 1295, 1294, 1293, 1292, 1291, 1290, 1289, 1288, 1287, 1286, 1285, 1284, 1283, 1282, 1281, 1280, 1279, 1278, 1277, 1276, 1275, 1274, 1273, 1272, 1271, 1270, 1269, 1268, 1267, 1266, 1265, 1264, 1263, 1262, 1261, 1260, 1259, 1258, 1257, 1256, 1255, 1254, 1253, 1252, 1251, 1250, 1249, 1248, 1247, 1246, 1245, 1244, 1243, 1242, 1241, 1240, 1239, 1238, 1237, 1236, 1235, 1234, 1233, 1232, 1231, 1230, 1229, 1228, 1227, 1226, 1225, 1224, 1223, 1222, 1221, 1220, 1219, 1218, 1217, 1216, 1215, 1214, 1213, 1212, 1211, 1210, 1209, 1208, 1207, 1206, 1205, 1204, 1203, 1202, 1201, 1200, 1199, 1198, 1197, 1196, 1195, 1194, 1193, 1192, 1191, 1190, 1189, 1188, 1187, 1186, 1185, 1184, 1183, 1182, 1181, 1180, 1179, 1178, 1177, 1176, 1175, 1174, 1173, 1172, 1171, 1170, 1169, 1168, 1167, 1166, 1165, 1164, 1163, 1162, 1161, 1160, 1159, 1158, 1157, 1156, 1155, 1154, 1153, 1152, 1151, 1150, 1149, 1148, 1147, 1146, 1145, 1144, 1143, 1142, 1141, 1140, 1139, 1138, 1137, 1136, 1135, 1134, 1133, 1132, 1131, 1130, 1129, 1128, 1127, 1126, 1125, 1124, 1123, 1122, 1121, 1120, 1119, 1118, 1117, 1116, 1115, 1114, 1113, 1112, 1111, 1110, 1109, 1108, 1107, 1106, 1105, 1104, 1103, 1102, 1101, 1100, 1099, 1098, 1097, 1096, 1095, 1094, 1093, 1092, 1091, 1090, 1089, 1088, 1087, 1086, 1085, 1084, 1083, 1082, 1081, 1080, 1079, 1078, 1077, 1076, 1075, 1074, 1073, 1072, 1071, 1070, 1069, 1068, 1067, 1066, 1065, 1064, 1063, 1062, 1061, 1060, 1059, 1058, 1057, 1056, 1055, 1054, 1053, 1052, 1051, 1050, 1049, 1048, 1047, 1046, 1045, 1044, 1043, 1042, 1041, 1040, 1039, 1038, 1037, 1036, 1035, 1034, 1033, 1032, 1031, 1030, 1029, 1028, 1027, 1026, 1025, 1024, 1023, 1022, 1021, 1020, 1019, 1018, 1017, 1016, 1015, 1014, 1013, 1012, 1011, 1010, 1009, 1008, 1007, 1006, 1005, 1004, 1003, 1002, 1001, 1000, 999, 998, 997, 996, 995, 994, 993, 992, 991, 990, 989, 988, 987, 986, 985, 984, 983, 982, 981, 980, 979, 978, 977, 976, 975, 974, 973, 972, 971, 970, 969, 968, 967, 966, 965, 964, 963, 962, 961, 960, 959, 958, 957, 956, 955, 954, 953, 952, 951, 950, 949, 948, 947, 946, 945, 944, 943, 942, 941, 940, 939, 938, 937, 936, 935, 934, 933, 932, 931, 930, 929, 928, 927, 926, 925, 924, 923, 922, 921, 920, 919, 918, 917, 916, 915, 914, 913, 912, 911, 910, 909, 908, 907, 906, 905, 904, 903, 902, 901, 900, 899, 898, 897, 896, 895, 894, 893, 892, 891, 890, 889, 888, 887, 886, 885, 884, 883, 882, 881, 880, 879, 878, 877, 876, 875, 874, 873, 872, 871, 870, 869, 868, 867, 866, 865, 864, 863, 862, 861, 860, 859, 858, 857, 856, 855, 854, 853, 852, 851, 850, 849, 848, 847, 846, 845, 844, 843, 842, 841, 840, 839, 838, 837, 836, 835, 834, 833, 832, 831, 830, 829, 828, 827, 826, 825, 824, 823, 822, 821, 820, 819, 818, 817, 816, 815, 814, 813, 812, 811, 810, 809, 808, 807, 806, 805, 804, 803, 802, 801, 800, 799, 798, 797, 796, 795, 794, 793, 792, 791, 790, 789, 788, 787, 786, 785, 784, 783, 782, 781, 780, 779, 778, 777, 776, 775, 774, 773, 772, 771, 770, 769, 768, 767, 766, 765, 764, 763, 762, 761, 760, 759, 758, 757, 756, 755, 754, 753, 752, 751, 750, 749, 748, 747, 746, 745, 744, 743, 742, 741, 740, 739, 738, 737, 736, 735, 734, 733, 732, 731, 730, 729, 728, 727, 726, 725, 724, 723, 722, 721, 720, 719, 718, 717, 716, 715, 714, 713, 712, 711, 710, 709, 708, 707, 706, 705, 704, 703, 702, 701, 700, 699, 698, 697, 696, 695, 694, 693, 692, 691, 690, 689, 688, 687, 686, 685, 684, 683, 682, 681, 680, 679, 678, 677, 676, 675, 674, 673, 672, 671, 670, 669, 668, 667, 666, 665, 664, 663, 662, 661, 660, 659, 658, 657, 656, 655, 654, 653, 652, 651, 650, 649, 648, 647, 646, 645, 644, 643, 642, 641, 640, 639, 638, 637, 636, 635, 634, 633, 632, 631, 630, 629, 628, 627, 626, 625, 624, 623, 622, 621, 620, 619, 618, 617, 616, 615, 614, 613, 612, 611, 610, 609, 608, 607, 606, 605, 604, 603, 602, 601, 600, 599, 598, 597, 596, 595, 594, 593, 592, 591, 590, 589, 588, 587, 586, 585, 584, 583, 582, 581, 580, 579, 578, 577, 576, 575, 574, 573, 572, 571, 570, 569, 568, 567, 566, 565, 564, 563, 562, 561, 560, 559, 558, 557, 556, 555, 554, 553, 552, 551, 550, 549, 548, 547, 546, 545, 544, 543, 542, 541, 540, 539, 538, 537, 536, 535, 534, 533, 532, 531, 530, 529, 528, 527, 526, 525, 524, 523, 522, 521, 520, 519, 518, 517, 516, 515, 514, 513, 512, 511, 510, 509, 508, 507, 506, 505, 504, 503, 502, 501, 500, 499, 498, 497, 496, 495, 494, 493, 492, 491, 490, 489, 488, 487, 486, 485, 484, 483, 482, 481, 480, 479, 478, 477, 476, 475, 474, 473, 472, 471, 470, 469, 468, 467, 466, 465, 464, 463, 462, 461, 460, 459, 458, 457, 456, 455, 454, 453, 452, 451, 450, 449, 448, 447, 446, 445, 444, 443, 442, 441, 440, 439, 438, 437, 436, 435, 434, 433, 432, 431, 430, 429, 428, 427, 426, 425, 424, 423, 422, 421, 420, 419, 418, 417, 416, 415, 414, 413, 412, 411, 410, 409, 408, 407, 406, 405, 404, 403, 402, 401, 400, 399, 398, 397, 396, 395, 394, 393, 392, 391, 390, 389, 388, 387, 386, 385, 384, 383, 382, 381, 380, 379, 378, 377, 376, 375, 374, 373, 372, 371, 370, 369, 368, 367, 366, 365, 364, 363, 362, 361, 360, 359, 358, 357, 356, 355, 354, 353, 352, 351, 350, 349, 348, 347, 346, 345, 344, 343, 342, 341, 340, 339, 338, 337, 336, 335, 334, 333, 332, 331, 330, 329, 328, 327, 326, 325, 324, 323, 322, 321, 320, 319, 318, 317, 316, 315, 314, 313, 312, 311, 310, 309, 308, 307, 306, 305, 304, 303, 302, 301, 300, 299, 298, 297, 296, 295, 294, 293, 292, 291, 290, 289, 288, 287, 286, 285, 284, 283, 282, 281, 280, 279, 278, 277, 276, 275, 274, 273, 272, 271, 270, 269, 268, 267, 266, 265, 264, 263, 262, 261, 260, 259, 258, 257, 256, 255, 254, 253, 252, 251, 250, 249, 248, 247, 246, 245, 244, 243, 242, 241, 240, 239, 238, 237, 236, 235, 234, 233, 232, 231, 230, 229, 228, 227, 226, 225, 224, 223, 222, 221, 220, 219, 218, 217, 216, 215, 214, 213, 212, 211, 210, 209, 208, 207, 206, 205, 204, 203, 202, 201, 200, 199, 198, 197, 196, 195, 194, 193, 192, 191, 190, 189, 188, 187, 186, 185, 184, 183, 182, 181, 180, 179, 178, 177, 176, 175, 174, 173, 172, 171, 170, 169, 168, 167, 166, 165, 164, 163, 162, 161, 160, 159, 158, 157, 156, 155, 154, 153, 152, 151, 150, 149, 148, 147, 146, 145, 144, 143, 142, 141, 140, 139, 138, 137, 136, 135, 134, 133, 132, 131, 130, 129, 128, 127, 126, 125, 124, 123, 122, 121, 120, 119, 118, 117, 116, 115, 114, 113, 112, 111, 110, 109, 108, 107, 106, 105, 104, 103, 102, 101, 100, 99, 98, 97, 96, 95, 94, 93, 92, 91, 90, 89, 88, 87, 86, 85, 84, 83, 82, 81, 80, 79, 78, 77, 76, 75, 74, 73, 72, 71, 70, 69, 68, 67, 66, 65, 64, 63, 62, 61, 60, 59, 58, 57, 56, 55, 54, 53, 52, 51, 50, 49, 48, 47, 46, 45, 44, 43, 42, 41, 40, 39, 38, 37, 36, 35, 34, 33, 32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1], "dtype": "int32"}], "e": 3999}, {"a": [{"__nd__": [6, 0, 11, 87, 11, 24, 0, 65, 9, 48, 25, 78, 10, 88, 35, 89, 49, 52, 64, 96, 16, 91, 73, 67, 37, 76, 19, 17, 92, 86, 12, 99, 85, 91, 62, 11, 76, 38, 73, 92, 92, 9, 91, 31, 13, 67, 73, 62, 2, 28, 15, 72, 88, 0, 64, 85, 54, 58, 95, 1, 17, 65, 87, 39, 14, 59, 81, 47, 9, 29, 93, 39, 68, 53, 62, 16, 12, 49, 12, 94, 38, 1, 45, 60, 76, 94, 28, 39, 61, 32, 44, 55, 83, 16, 93, 3, 38, 80, 42, 53, 77, 92, 67, 33, 36, 64, 74, 17, 21, 54, 12, 20, 77, 32, 85, 36, 40, 29, 31, 99, 32, 45, 77, 2, 32, 66, 5, 31, 99, 56, 76, 26, 3, 11, 73, 22, 13, 52, 18, 94, 20, 93, 9, 84, 91, 24, 55, 97, 65, 56, 77, 64, 78, 14, 48, 43, 70, 26, 32, 97, 66, 19, 37, 35, 77, 41, 30, 47, 74, 33, 76, 54, 9, 92, 6, 87, 42, 73, 9, 68, 87, 19, 10, 53, 37, 98, 15, 18, 37, 29, 43, 68, 16, 46, 49, 70, 58, 52, 64, 1, 56, 10, 35, 39, 62, 84, 39, 48, 32, 30, 47, 76, 14, 32, 25, 54, 26, 69, 24, 54, 81, 55, 55, 95, 17, 96, 49, 92, 51, 61, 91, 2, 9, 35, 21, 92, 96, 84, 92, 3, 8, 47, 85, 9, 92, 90, 66, 1, 17, 76, 16, 22, 57, 23, 5, 65, 6, 64, 1, 92, 15, 96, 55, 52, 26, 19, 52, 95, 55, 76, 31, 13, 30, 59, 64, 1, 72, 0, 66, 27, 36, 29, 52, 85, 3, 50, 7, 20, 85, 39, 41, 49, 58, 63, 55, 58, 37, 39, 71, 72, 32, 90, 14, 85, 25, 65, 68, 58, 50, 72, 69, 12, 83, 24, 53, 87, 1, 25, 49, 79, 8, 63, 93, 59, 0, 26, 85, 58, 54, 45, 25, 61, 23, 23, 79, 37, 93, 96, 21, 35, 25, 35, 58, 5, 57, 88, 78, 31, 32, 60, 84, 24, 73, 24, 2, 41, 66, 95, 64, 10, 22, 92, 99, 65, 45, 13, 39, 28, 99, 83, 42, 67, 78, 89, 83, 19, 85, 25, 78, 22, 46, 65, 69, 35, 90, 53, 50, 50, 29, 82, 73, 9, 98, 66, 70, 80, 7, 33, 20, 31, 59, 0, 91, 85, 92, 13, 5, 63, 50, 86, 83, 61, 90, 76, 88, 46, 73, 40, 78, 66, 13, 57, 15, 6, 42, 25, 27, 66, 84, 6, 79, 41, 32, 36, 10, 22, 20, 96, 17, 38, 85, 93, 32, 17, 4, 81, 95, 4, 6, 1, 86, 48, 40, 95, 87, 89, 24, 59, 76, 54, 57, 34, 61, 16, 33, 99, 34, 37, 98, 46, 76, 60, 13, 31, 56, 11, 54, 93, 83, 33, 55, 88, 18, 99, 2, 23, 0, 84, 93, 8, 7, 22, 50, 72, 26, 9, 14, 32, 20, 93, 95, 33, 71, 81, 20, 63, 1, 68, 41, 66, 45, 70, 5, 26, 56, 37, 8, 25, 6, 61, 68, 25, 63, 17, 50, 56, 29, 42, 8, 55, 7, 52, 3, 28, 15, 36, 14, 2, 72, 65, 36, 76, 21, 22, 28, 58, 65, 1, 98, 55, 16, 3, 48, 14, 7, 30, 87, 58, 88, 92, 24, 73, 47, 16, 60, 5, 85, 37, 41, 97, 40, 76, 56, 65, 40, 81, 87, 42, 25, 17, 33, 91, 4, 28, 55, 8, 84, 17, 46, 22, 31, 6, 85, 32, 98, 70, 85, 78, 79, 34, 16, 55, 24, 27, 43, 71, 3, 78, 40, 46, 25, 82, 8, 45, 16, 60, 57, 94, 38, 0, 59, 42, 2, 46, 9, 41, 15, 39, 94, 67, 94, 54, 52, 0, 38, 36, 41, 72, 23, 12, 31, 74, 8, 88, 23, 80, 87, 16, 28, 71, 47, 58, 63, 83, 33, 37, 32, 73, 17, 26, 8, 28, 77, 68, 52, 51, 27, 79, 84, 93, 22, 81, 20, 23, 72, 32, 80, 27, 28, 27, 68, 55, 60, 10, 72, 68, 7, 19, 26, 64, 47, 48, 92, 2, 33, 4, 86, 49, 38, 34, 4, 80, 21, 38, 7, 47, 85, 38, 95, 65, 10, 65, 23, 79, 25, 64, 44, 35, 96, 66, 40, 24, 65, 59, 7, 97, 76, 99, 73, 26, 11, 49, 10, 73, 4, 59, 94, 71, 24, 65, 41, 76, 69, 48, 68, 36, 32, 75, 34, 65, 97, 71, 60, 67, 82, 46, 68, 56, 82, 32, 2, 83, 60, 56, 36, 92, 23, 14, 19, 35, 88, 10, 3, 47, 91, 20, 5, 17, 66, 88, 8, 66, 3, 7, 27, 78, 36, 66, 69, 74, 96, 47, 80, 0, 82, 93, 13, 51, 69, 33, 19, 60, 70, 23, 83, 42, 26, 11, 86, 4, 51, 62, 45, 6, 69, 92, 8, 21, 12, 29, 0, 85, 9, 15, 48, 83, 80, 64, 59, 22, 49, 55, 87, 99, 96, 44, 50, 81, 89, 4, 29, 65, 99, 90, 30, 71, 90, 50, 39, 5, 58, 17, 2, 44, 63, 43, 25, 45, 54, 85, 57, 49, 48, 30, 56, 74, 70, 13, 26, 37, 61, 97, 92, 2, 93, 96, 6, 4, 91, 52, 38, 29, 66, 74, 61, 25, 67, 38, 24, 70, 22, 83, 98, 37, 60, 91, 47, 67, 87, 49, 64, 35, 91, 94, 69, 78, 86, 52, 28, 96, 48, 58, 72, 94, 20, 36, 19, 0, 60, 7, 31, 95, 47, 15, 38, 15, 53, 67, 23, 14, 49, 1, 63, 41, 28, 29, 57, 98, 24, 96, 3, 96, 81, 93, 97, 58, 76, 4, 5, 74, 21, 22, 88, 20, 58, 19, 28, 24, 55, 21, 14, 96, 58, 39, 15, 53, 61, 50, 17, 0, 52, 7, 20, 62, 98, 65, 18, 31, 58, 60, 44, 95, 94, 79, 43, 38, 64, 6, 36, 70, 79, 38, 58, 98, 38, 56, 30, 13, 39, -7, 42, 73, 34, 26, 26, 30, 77, 77, 37, 90, 99, 60, 84, 24, 86, 16, 59, 1, 44, 97, 26, 19, 17, 83, 82, 63, 65, 17, 99, 60, 58, 17, 36, 66, 77, 93, 89, 14, 36, 81, 27, 42, 25, 56, 32, 32, 70, 68, 61, 17, 87, 20, 52, 30, 88, 84, 19, 51, 86, 71, 29, 4, 62, 13, 85, 90, 65, 91, 70, 29, 89, 11, 50, 37, 9, 44, 51, 33, 56, 34, 65, 45, 83, 35, 90, 75, 17, 57, 86, 92, 74, 95, 10, 69, 49, 83, 16, 83, 53, 83, 33, 24, 29, 70, 69, 66, 35, 27, 43, 71, 17, 85, 89, 61, 86, 58, 62, 3, 35, 46, 95, 59, 5, 83, 13, 56, 60, 30, 90, 21, 96, 55, 6, 33, 69, 74, 38, 52, 69, 89, 76, 93, 55, 66, 73, 25, 59, 16, 33, 16, 9, 29, 97, 87, 40, 52, 35, 50, 73, 61, 19, 10, 6, 4, 65, 53, 42, 22, 16, 1, 69, 32, 69, 34, 5, 82, 87, 40, 77, 57, 65, 72, 94, 40, 8, 64, 59, 23, 3, 15, 83, 75, 65, 81, 96, 51, 13, 10, 7, 3, 29, 40, 66, 78, 9, 76, 97, 68, 22, 58, 53, 44, 22, 44, 45, 96, 64, 92, 28, 81, 44, 52, 8, 81, 46, 91, 10, 80, 9, 39, 41, 58, 9, 70, 20, 95, 41, 15, 57, 76, 98, 23, 31, 44, 58, 88, 47, 43, 83, 83, 80, 0, 28, 48, 10, 69, 16, 69, 82, 38, 96, 48, 75, 87, 3, 73, 8, 24, 60, 13, 38, 75, 76, 22, 70, 54, 16, 96, 61, 76, 78, 40, 38, 81, 55, 23, 28, 81, 13, 6, 17, 6, 36, 34, 84, 51, 6, 62, 63, 15, 65, 27, 15, 91, 73, 85, 6, 19, 95, 57, 8, 38, 55, 93, 61, 47, 85, 25, 98, 3, 51, 92, 46, 73, 22, 90, 28, 93, 40, 63, 20, 16, 87, 93, 20, 73, 17, 19, 75, 62, 68, 31, 20, 62, 65, 32, 99, 57, 95, 9, 53, 44, 9, 53, 3, 12, 29, 71, 82, 28, 23, 45, 6, 31, 37, 34, 48, 66, 65, 17, 94, 21, 49, 66, 5, 96, 39, 84, 32, 94, 58, 92, 62, 75, 31, 96, 18, 47, 50, 75, 94, 27, 92, 78, 78, 95, 43, 18, 90, 73, 62, 8, 12, 52, 7, 34, 8, 11, 24, 6, 57, 79, 21, 4, 82, 29, 66, 47, 43, 11, 26, 85, 33, 57, 29, 26, 2, 42, 85, 57, 96, 81, 34, 66, 48, 57, 52, 27, 15, 52, 82, 87, 27, 85, 34, 41, 79, 14, 50, 44, 71, 24, 92, 18, 63, 9, 24, 68, 32, 79, 91, 44, 87, 53, 52, 7, 77, 35, 74, 38, 38, 75, 89, 17, 4, 63, 26, 22, 91, 29, 10, 62, 47, 12, 95, 70, 35, 31, 66, 78, 39, 15, 88, 81, 83, 50, 10, 48, 32, 32, -7, 38, 68, 96, 10, 7, 15, 76, 85, 27, 98, 45, 2, 28, 61, 47, 63, 87, 86, 54, 66, 77, 18, 31, 52, 40, 93, 46, 63, 77, 37, 24, 17, 14, 59, 39, 56, 59, 89, 64, 44, 53, 62, 4, 21, 25, 21, 39, 0, 82, 95, 97, 83, 0, 0, 11, 6, 22, 72, 81, 27, 91, 99, 3, 93, 66, 12, 95, 84, 48, 95, 5, 26, 81, 52, 49, 42, 0, 61, 81, 96, 31, 63, 57, 41, 62, 87, 81, 98, 89, 75, 0, 83, 65, 84, 57, 78, 72, 91, 71, 20, 65, 90, 68, 11, 74, 28, 53, 53, 91, 33, 54, 62, 51, 17, 59, 87, 80, 10, 89, 0, 79, 8, 95, 26, 15, 69, 39, 42, 73, 58, 11, 3, 42, 87, 10, 14, 79, 8, 96, 11, 5, 41, 80, 75, 40, 86, 19, 87, 3, 22, 76, 88, 55, 92, 7, 12, 35, 95, 0, 72, 50, 81, 92, 1, 64, 90, 67, 63, 29, 40, 79, 95, 88, 39, 88, 23, 46, 33, 46, 10, 26, 72, 17, 91, 82, 17, 6, 38, 83, 19, 86, 37, 43, 39, 80, 41, 57, 67, 35, 49, 3, 77, 56, 32, 84, 46, 36, 49, 36, 23, 55, 95, 95, 57, 3, 36, 53, 69, 97, 4, 46, 55, 89, 3, 31, 24, 20, 19, 17, 20, 21, 47, 49, 67, 54, 78, 14, 55, 65, 3, 62, 22, 39, 59, 96, 59, 23, 3, 65, 67, 58, 5, 94, 27, 82, 71, 35, 17, 82, 36, 86, 48, 97, 79, 62, 82, 38, 57, 11, 35, 50, 62, 90, 95, 37, 19, 11, 58, 27, 29, 3, 9, 50, 22, 80, 58, 39, 60, 49, 49, 64, 76, 36, 43, 8, 88, 63, 28, 73, 50, 11, 51, 69, 64, 48, 89, 39, 94, 8, 20, 26, 8, 86, 58, 83, 37, 10, 18, 58, 4, 65, 25, 98, 30, 44, 66, 66, 54, 27, 45, 95, 47, 87, 34, 9, 76, 62, 28, 37, 80, 41, 27, 25, 8, 70, 3, 60, 12, 6, 15, 3, 94, 53, 69, 81, 56, 86, 82, 97, 12, 11, 70, 82, 1, 18, 74, 53, 97, 60, 41, 14, 37, 22, 83, 74, 33, 82, 9, 71, 32, 27, 79, 95, 31, 23, 3, 17, 86, 83, 78, 44, 36, 11, 9, 75, 75, 61, 57, 22, 79, 45, 67, 89, 83, 63, 34, 72, 86, 8, 77, 51, 21, 11, 32, 61, 6, 46, 36, 40, 74, 23, 58, 28, 39, 29, 9, 92, 83, 9, 45, 13, 96, 11, 74, 82, 77, 73, 96, 42, 33, 41, 98, 58, 84, 80, 48, 7, 15, 96, 39, 88, 35, 91, 41, 8, 47, 47, 28, 86, 79, 61, 89, 38, 61, 85, 31, 99, 90, 71, 85, 98, 7, 38, 78, 92, 7, 43, 17, 37, 0, 36, 22, 70, 64, 13, 52, 90, 51, 81, 30, 25, 12, 11, 19, 68, 29, 40, 89, 81, 61, 44, 32, 42, 26, 55, 44, 4, 73, 10, 45, 85, 22, 31, 43, 63, 14, 12, 79, 4, 65, 94, 29, 69, 50, 15, 97, 57, 97, 54, 2, 2, 60, 74, 2, 11, 72, 69, 87, 14, 2, 54, 23, 41, 4, 27, 50, 66, 71, 80, 46, 62, 50, 31, 83, 97, 32, 71, 90, 71, 95, 49, 23, 95, 94, 87, 0, 86, 10, 25, 32, 33, 58, 93, 71, 97, 24, 22, 40, 37, 1, 42, 57, 55, 92, 47, 38, 94, 79, 55, 84, 14, 31, 85, 65, 57, 78, 50, 81, 68, 16, 83, 60, 65, 64, 72, 86, 42, 64, 16, 3, 20, 8, 35, 92, 16, 53, 51, 56, 76, 5, 78, 48, 63, 77, 56, 54, 42, 10, 7, 52, 47, 19, 73, 17, 51, 34, 38, 34, 41, 23, 66, 41, 40, 76, 11, 35, 8, 6, 39, 88, 4, 19, 73, 65, 41, 38, 17, 28, 7, 3, 49, 0, 8, 2, 79, 98, 77, 22, 22, 22, 4, 73, 66, 14, 98, 89, 49, 23, 88, 53, 14, 26, 57, 43, 56, 14, 38, 82, 89, 31, 29, 79, 4, 87, 17, 21, 37, 33, 82, 86, 68, 89, 43, 45, 31, 75, 61, 14, 57, 61, 78, 19, 99, 94, 92, 53, 5, 13, 18, 92, 68, 96, 47, 8, 98, 21, 79, 14, 28, 99, 4, 73, 47, 69, 78, 46, 67, 64, 26, 83, 72, 32, 61, 98, 67, 68, 30, 28, 52, 2, 89, 76, 22, 92, 26, 69, 64, 55, 5, 82, 3, 15, 60, 97, 57, 49, 94, 96, 47, 78, 13, 57, 75, 90, 29, 36, 7, 56, 67, 72, 27, 5, 74, 41, 69, 90, 86, 79, 51, 52, 79, 72, 16, 86, 1, 55, 9, 51, 58, 92, 72, 61, 25, 80, 57, 43, 5, 18, 4, 63, 40, 43, 58, 46, 14, 32, 5, 57, 3, 79, 95, 56, 6, 61, 11, 27, 23, 70, 29, 81, 6, 0, 66, 75, 25, 14, 61, 73, 0, 63, 92, 32, 4, 82, 49, 49, 15, 37, 58, 7, 83, 59, 38, 36, 70, 78, 41, 67, 4, 83, 28, 85, 47, 52, 12, 93, 15, 28, 19, 94, 81, 85, 36, 60, 71, 44, 69, 26, 14, 6, 91, 74, 2, 80, 61, 92, 13, 40, 95, 36, 99, 69, 25, 86, 87, 52, 42, 66, 17, 46, 6, 23, 53, 39, 85, 78, 0, 74, 6, 41, 60, 0, 24, 48, 15, 78, 77, 99, 40, 31, 50, 27, 50, 12, 11, 45, 47, 98, 1, 54, 7, 99, 56, 69, 81, 72, 7, 24, 35, 79, 54, 26, 84, 34, 95, 48, 9, 95, 68, 84, 95, 43, 22, 53, 78, 37, 32, 31, 81, 60, 16, 18, 6, 81, 65, 92, 92, 21, 22, 79, 27, 39, 16, 46, 73, 24, 81, 35, 17, 12, 64, 41, 76, 11, 34, 89, 16, 66, 65, 21, 12, 44, 48, 35, 5, 52, 13, 15, 7, -7, 55, 79, 1, 26, 26, 55, 32, 12, 0, 22, 20, 55, 62, 60, 11, 48, 70, 80, 19, 60, 61, 59, 23, 97, 2, 11, 11, 75, 13, 27, 92, 65, 58, 25, 95, 33, 40, 60, 0, 27, 32, 73, 77, 5, 9, 46, 34, 83, 78, 57, 60, 59, 53, 50, 22, 69, 62, 92, 98, 0, 56, 63, 74, 60, 92, 32, 42, 5, 61, 12, 53, 30, 95, 90, 42, 49, 54, 61, 22, 2, 93, 68, 33, 89, 22, 80, 44, 54, 8, 97, 44, 25, 62, 75, 58, 54, 1, 92, 74, 69, 23, 56, 0, 50, 59, 25, 61, 78, 46, 47, 25, 60, 28, 12, 0, 16, 12, 41, 26, 36, 66, 2, 66, 81, 28, 76, 83, 70, 24, 97, 5, 25, 27, 99, 36, 72, 75, 94, 3, 9, 51, 17, 85, 43, 67, 77, 76, 3, 69, 46, 0, 63, 52, 26, 75, 8, 15, 83, 43, 25, 2, 80, 52, 45, 98, 3, 3, 88, 22, 83, 19, 47, 50, 34, 29, 21, 96, 79, 24, 20, 79, 84, 57, 1, 10, 31, 53, 96, 60, 89, 92, 60, 95, 65, 9, 11, 5, 1, 50, 48, 4, 11, 83, 70, 89, 75, 44, 40, 78, 23, 40, 64, 87, 85, 89, 87, 16, 50, 50, 94, 15, 44, 7, 57, 2, 0, 18, 94, 32, 4, 63, 47, 29, 37, 78, 91, 44, 67, 77, 80, 33, 9, 51, 40, 24, 5, 78, 61, 24, 22, 52, 45, 7, 42, 36, 82, 65, 75, 44, 90, 54, 98, 56, 43, 33, 41, 18, 36, 5, 28, 99, 24, 79, 30, 39, 29, 94, 65, 13, 74, 35, 67, 68, 63, 96, 88, 82, 75, 28, 89, 5, 4, 10, 46, 45, 84, 36, 62, 21, 64, 31, 41, 4, 98, 20, 50, 73, 66, 59, 79, 38, 30, 18, 83, 55, 85, 28, 38, 37, 72, 41, 19, 55, 19, 79, 20, 27, 47, 39, 24, 17, 27, 59, 79, 41, 93, 67, 5, 29, 72, 57, 58, 51, 50, 4, 40, 41, 38, 86, 42, 75, 24, 92, 18, 26, 8, 69, 39, 81, 6, 22, 52, 76, 60, 6, 72, 85, 23, 23, 74, 19, 65, 50, 9, 25, 92, 87, 10, 12, 37, 90, 55, 49, 28, 97, 15, 80, 74, 61, 80, 1, 57, 32, 46, 51, 88, 11, 95, 61, 0, 15, 64, 2, 22, 87, 19, 83, 37, 52, 2, 40, 10, 19, 8, 42, 72, 37, 68, 12, 62, 37, 57, 46, 54, 76, 9, 42, 35, 38, 70, 71, 57, 63, 28, 98, 79, 26, 78, 72, 15, 21, 58, 31, 4, 50, 46, 4, 91, 8, 7, 23, 13, 61, 22, 33, 18, 73, 89, 58, 68, 35, 90, 83, 94, 68, 56, 64, 30, 34, 80, 23, 80, 86, 10, 24, 64, 25, 65, 6, 98, 57, 7, 16, 20, 21, 4, 17, 96, 64, 59, 63, 19, 26, 70, 4, 87, 47, 71, 65], "dtype": "int32"}], "e": 999}, {"a": [{"__nd__": [992157317, -204339716, -621944975, -396301746, -993792861, -516708377, -932931412, -640591754, 980960244, 719079387, 228523676, -261686439, -950893053, 344373939, 630678711, 370064351, 464442833, -446992815, 733300691, -639479908, 362052345, 956895282, 106823603, -212329701, -645048754, -162109300, 375260298, 757347643, -865942635, 360889466, -33395477, -961064561, -977989486, 247777198, 136031116, 879367944, 255851917, 988838688, 52459024, 102257462, -746922114, 363612499, 257577192, -657254699, 164240954, 332720218, 537627081, 842159939, -794164359, 134243424, -342044680, -134383853, -156009896, 138196887, 548171279, -924488455, -559195802, -127993291, -459550866, -133792985, -381850582, -317245507, -732560679, 425638681, -497912230, -737760392, 351323144, -725649412, -173409099, -533425595, 334284888, -458894056, 301590083, 648442815, -124999106, 128261218, -776117157, 746743590, 605877074, 187875242, 134970304, -819824157, -403136285, -406954525, 83893284, 694114549, 281157169, 636127116, 455970969, 878377586, -357272811, -638465885, 533691328, 249732699, -139461596, -381786137, 426188718, 516680067, 350289886, 954617646, -566637153, 177399321, -614709004, -715064820, 391856559, -210109043, -675540108, -963489801, -4126276, -587044903, 505875677, -763833534, 803730534, 137040691, -173203295, -161477421, -329385219, 608944577, -392463600, 245162657, -259612450, -579836814, -95359451, -664175715, 658721272, -272735231, 367646770, 81316430, -211691548, -532051158, 573182845, -798794992, 720314683, 847517221, -19361174, 865609963, -596850156, 274289310, 253379567, -886181329, -438656953, 426706899, -825496863, 882730571, 186039488, -181265792, 679847133, 367506082, -327378229, 163193207, -19738488, 752908482, -671459331, 937850987, 457584417, 585560184, -591785545, 618393353, -59086062, 467325895, -285465015, -779214847, 139509385, -466869853, -2728641, 540843394, -416192439, 834635703, 266855174, -25227420, -523029751, -857093194, -765320946, 138807337, 575783905, -424684217, -998796910, 221102773, 774719148, 190584909, -226230984, -394087540, 678009631, 457789358, 744165610, 312251253, 725074711, -312099063, -636628863, 398922886, -943125808, 591476505, 333028017, -662474581, -51614691, -571183542, -91860243, 360872828, -695701631, -893351053, 670733939, -56978841, -865009322, 542454159, -71718859, 324597103, -512342867, 829550692, -59609090, 764999386, -626404676, -847224298, -809211565, -626648893, 618345054, 18370501, -7988577, 244183998, -443645953, 758899, 433055014, -979441675, -113188626, -192375683, 907115701, -282899633, -476243632, 468326922, 248398973, 75546756, 199126563, -506804555, 604119299, 281897769, -127648109, 685348500, -90092517, 407278047, -635785430, -691424852, -231125541, 672507784, -961897154, -650914378, -152063323, 942466721, -765701309, -709739270, 536762134, 470863071, 164668939, 492934892, 95265837, -383834558, 500264810, -655267201, 669899156, 486317770, 646941083, -413613090, 814125872, 586046140, 8166064, -135832932, -481018256, -456350818, -1931718, -393854140, 499521038, 417785927, 400580883, 398315881, 823685494, 173696207, 112858594, -362339721, 289090989, 638711810, -997276098, 388671284, -663124585, 927319879, 267532216, -625676468, -58864119, -141150927, 83411479, 814801984, 237936886, -794030544, 729007398, 395726801, -461019887, 586349312, 10939780, 788774182, 718226957, -476722854, 592780441, 958303312, -520305854, 457122347, 654364965, 408176824, 324488494, 796730523, -771979403, 778318905, 548691944, 914754328, -908379494, 482572611, 450290316, -431035960, -506783528, 263930045, -649679610, -99702686, 6752272, -906538407, -772380471, -399229905, -912356659, -353327117, -773104262, 72483211, -150648585, -381266195, 38029708, -447761724, -289482634, -926686180, -383602005, -304117043, -290403423, 486150098, 289973868, 354824625, 636943322, -591849242, -215030267, -228290487, 362893856, -828822350, -163798625, -522245449, -658826309, 982533833, 49627340, -793273750, 906915364, 955030084, 344123670, -712625097, 876800852, 163125018, -844725517, 432790945, -65815479, -151454711, 739608238, -470138949, -213458052, 751599988, 962920654, -138976614, 432278891, 123499996, 340504305, -726873643, 752590609, -70865043, -724420782, -441974298, -211941070, -137557168, -85743957, -921057405, -55179258, -391510770, -482265005, -841816586, 63222827, -360169382, 592542008, 766892584, 811759039, 578782777, -567265226, 268882920, -945125601, 561216, 329055301, 875747609, 739733564, 614739744, 698482248, -357469927, 891803499, 299055193, 234399520, -918825993, 569969770, -452046444, 862000006, 719251049, -309486262, -37460540, 468891090, -622149501, 548257396, 681406242, 255919039, 496695224, -975168960, -176655593, 261701043, -525373990, 795875715, 98690278, -879402679, 564017411, 781204013, 43790989, 887360523, -682355601, 915714782, 865373892, -493092810, 611421807, -505038276, 655882064, -903721714, 218067541, 497801747, 703812152, -105354833, 85213223, -321322938, 572586772, -198222806, 955526470, 256540683, 144480002, -417058035, 634026680, -19318800, 965779860, -336281927, 426537395, -392831097, -707136634, 708736585, 733839444, -24124041, -894501491, 436496360, 557302483, -222800310, -578649925, 653559319, -103921310, -565515427, 118327105, 864627006, 55527556, 27544136, -581781529, -853511525, -731752769, -943235000, -799350644, -834124262, 477434986, -223768393, -678673431, 460837693, -743942742, -982364863, 581213307, -347054962, -481402119, 559852471, -719877637, 465692781, 84120241, 16115248, -154075857, 700372327, -442410252, 932580204, 70505180, -89258755, 685948344, 143420546, -651806250, -949309341, 847301749, 905636108, 215688489, -300846482, 48831091, 641155932, -969646930, 72539411, -924987155, 756560413, 167809985, 851333476, -169950159, -587846851, 140879286, -725467422, 411178174, 194549294, 225975648, 612012412, -579176920, 624207012, 385364400, -798464797, 473784940, 834092444, 958209009, -734643465, -471451338, -672632741, 881689719, 675293593, 30345942, 349183802, 173093935, -475374432, 256663348, 685427479, 26591608, 181899287, -672534402, 94868823, 752436410, -859196227, 316197758, 540072061, 582290315, -808996755, 977870405, -31994019, -809702672, -391174773, -304430831, 482496722, -309695698, -698263520, -271755388, -899469046, -434884825, -471500172, 539877819, -418931104, -820596219, -135392839, -984941380, 578432977, 79417791, 66413360, 864924743, 750686619, 971549886, -735788675, 209766299, -532781044, -913057993, -249389450, 887541809, 436029506, 34359823, 920931508, -850427706, -637562381, -315839175, -601500566, -877631161, -809600251, -881441808, 621818214, 897144949, -212584750, 605227920, -734402544, -199358616, -387401729, 506395152, -809574766, 546256535, 816032382, 779081056, 206007580, -557211092, 242181038, 741566443, 770048669, 763417560, 673717497, 432469673, -401716715, -149609486, -521465577, -350444135, 492130361, 520292960, 154318877, 114703148, 624283264, 822449707, -785499674, -821628595, 873641410, -407175505, -476998644, -581292319, 827751615, 154551215, -382362804, 280159843, -128138119, 322782206, -408373838, -847784801, 622139572, -271837148, 360723008, 685887505, 740358962, -113658201, -612171906, -274307095, -547626122, 670216496, -759470823, -337923772, 946905606, -932276786, -819822493, 795224777, -210957664, -406268160, 634194282, -965919756, -721524925, -700587472, 87995741, -189005456, 694735038, 664610543, -326670659, 673373369, 705614575, -784616790, 224133008, -603755231, -902758420, -39796785, 233242664, -558560512, 774603327, 769879145, -370925360, 20969179, 991637675, 441100022, 350139826, -523814172, 269537272, 315391132, 504899452, 757477195, -219870547, -621818567, -58549345, -395803942, 771691270, 94710512, -229611994, 284371348, 711590541, 564789655, 288521932, -731727915, -7017671, 3822985, -817576168, 419829060, -904505214, -441327578, -215619454, 233130515, 983952730, -392202473, -899754268, -256965228, -682414196, -744185367, -144725313, 190237618, -479043279, -766418794, -473976279, 347393993, -135282722, 606242566, 713048076, 526692428, 873027589, 67476871, -991839345, -968412475, -300714618, 333793673, -719765533, 18854470, -293356075, 394008022, 147778991, -978412162, 664226877, -351752422, 551870860, -648870838, 410860904, -722401781, -393679847, 926122233, -253453382, 641442991, 630199687, -217560119, 651002662, -706586748, 922841867, 73347154, -855462016, -514300911, 291373861, -634450104, -130088093, 357765375, 509012859, -277083460, 425059230, -954573279, -872259413, 644962101, 185854511, 830744044, -730848765, -410463107, -630275647, 919483241, -503949275, 438158395, 846919855, -382330579, -990394293, -19912657, 739352486, -113310220, -385618287, -359164271, -841843795, -11108013, 899797087, 239353676, 993202324, -75985728, 344428750, 470773232, 27121279, -3926546, 433467855, -373181876, -930059567, 101209818, -601729899, 140803415, -673382675, 717869609, -557967671, 560416712, -536920109, 404388693, -424333574, 263262586, -467764349, -933847934, 870226288, -606674484, 492751243, 675292194, -52974569, 71194131, -118500872, -902118549, 463415631, 405623518, 78943651, -298231441, 684033124, 965699565, 401988663, -484897023, -941361022, -812018388, -82826417, 503584844, 34971847, -198094131, 541080040, 636208070, -832773433, 683293003, -490501238, 45223669, 12955433, -2256535, 519892464, -801682355, 519422769, -648149714, 740815198, -719825065, -532612804, -139506822, 205781919, -631608223, -529023594, -425185556, 417680741, -408998537, -936365421, 807993088, -242335648, 511514937, 618836592, 489453480, 586274880, 334480867, 971016430, 946438275, -828490118, -28924593, 432849557, 978939756, 722160132, -641807366, 667809179, -995066668, -210998240, 409500369, -240809985, 897381365, 327240986, 249176445, -797124719, 72949738, -856247183, 699087600, -99385480, -296650104, 344269402, -659481142, 625267578, -855534514, 616102135, -347425303, 625879145, -797293761, 650104066, -423822944, -533423425, -74048945, 988835113, -170508403, 732475935, 127364051, -734603132, 660830157, 903978413, 718647369, -781385661, 141665691, 862173705, 960885775, -73461473, 994971017, 16532007, -882330031, 578773649, 937671511, 338055154, -48699299, 97605026, 444333069, 797089442, -336374831, -708733965, 490438973, -829473396, -658994771, 946979180, -825315852, -812123174, -492230646, 327992872, -181620656, -7433614, -504454206, -723703626, 565339347, -572394212, -902139469, -459584534, -190592246, 302581204, -950364390, -173646415, -41180373, -482085753, -475965655, 637149474, 513160153, 568706299, -746964988, 256448127, 105040812, -851331069, 146855290, 615191160, -16137451, -672982034, -627455577, -582005862, 702856173, 561500802, 716218596, -419610873, 588466508, 194170422, -925000861, -106394707, -848597814, 917559991, -833180012, 867507842, 864561809, 456519211, -217940086, 48001441, 354380895, -857498123, -542308637, 402101035, 736901621, 99856439, 523171987, -462503224, -903943541, 147191253, 171868968, 250478875, -633518419, 464484618, -7900066, 627438882, -158453718, 266569052, -562527807, -735077463, -911399518, 590814337, -695714634, 846233020, -552010720, -750449274, 405012358, 333668663, -556730066, 774524176, 195792024, -213336222, 376504035, -471555525, 696826609, 965799994, 528257574, -167906066, 474759637, 11556327, 809053093, -398362546, 679886263, 450776207, 116014317, -860631560, -418902668, 700937509, -300665780, 108748919, -835382919, 653733100, 581329503, 348502276, 341236323, -236576644, 556024418, -867680491, -818593332, 109541880, 814635955, 213694630, 880194192, -99379271, 449252421, -132025898, 320788265, -287404929, -804741635, -935799751, 474594173, -908583131, -251322358, 343177609, -692093700, 875957787, -994406271, -570585936, 141165713, -63206092, 804762567, 452833319, -458082893, 97577318, 735608124, -678566921, 569686834, 644949881, 233198191, 935448130, -831614948, 900963259, 606053401, 431598796, 983150637, 844660562, 279193185, 212642135, -484826070, -989868062, 243744943, -714651945, 69850193, 882328483, 592326019, -21881718, 901210821, -605583955, -765386865, -547834923, 391392412, -110609119, 395299861, 550836861, 809567681, -371940150, -908869657, 426171064, 906976378, -753587297, 14435546, 892502213, 797764260, 467463052, 933352891, 647752069, -268939070, -268699147, -318170888, 343867237, 203548450, 757410015, 424938788, 257720066, -220796317, 596451963, -388583668, -975440308, -496336987, 422801340, -748817768, -43911605, 407364437, 448838654, -550506019, -526703721, 893799479, 61937927, 510360537, -851651024, 489430750, -900634044, -72361714, -311545632, 65298600, 585687824, -626896866, 826015351, 504914663, 122551632, -507300701, 435285618, 232139129, 449972121, 116746167, 217761387, 46944655, -211645288, 548921734, 1516741, -131943789, -843490663, 735446166, -312899367, 308196433, -769090367, -349232375, -850797974, 808113983, 881347157, 5865430, -961815990, -404391548, -22931464, 249998134, -847216812, -233964427, -566752876, 868657677, -980093696, 663652560, -752952743, 207836507, 958941372, 767143955, 657056782, -106391736, 409187854, 983336097, -656810581, -331492842, 893428325, 179768177, -503148810, 368428936, 12872821, -510644247, 470677922, 968665711, -113370501, 737002501, -704542850, -640462973, 61196314, -332294341, -389471280, -491042673, 986699681, 15852857, -127573924, -57497269, -332616447, -98513965, 329132801, -103087728, 671194301, 313677912, -695186471, 784850067, -150913512, -178421242, -170913744, -408307396, 940992493, -903214812, -156759779, -961887303, 450810492, -573523529, 25467000, -553276922, 336178636, -564489568, 223761490, -477918418, 45299075, -840590133, -979089481, 222076323, -347583816, -568207569, -866272432, -526250672, -944069651, -412890333, 538761213, 428490795, -238731445, -384858614, 368364351, 267444740, -904594313, 974679619, 914876933, 117862599, -628479151, 113386988, -766241668, 18784356, -948321735, -401138329, 979229862, 731508021, -44454332, -319445373, 552132786, 189446712, -289730912, -290185930, -358577618, -169494642, -534061249, 107433065, -374006352, 595861135, 441675963, 137152379, 267682913, 985468647, -418340943, -598836720, 131789574, 544627673, 985657340, 835547390, -623608567, 792513813, 789904410, -848461002, -609607568, 227199185, 875686907, 130667017, 769269860, 898353187, 7032996, 632510578, 540213617, 641715328, -151270632, -999109419, -898809730, -968928058, -717548323, -367427872, -478474776, -197041100, -268102695, -697213226, -656620030, -552613604, 949333782, 134226425, -609958464, -16781285, 399721022, -964768125, -188269847, -803027163, 325250715, 819127309, 835911758, 589297184, -900464488, 189178802, 11915056, 138645049, 8532036, -783176500, -680307882, 519105276, 838005810, -189417556, -915103077, -458634207, 203921106, 118397819, 533252829, 168287113, -503670575, 133910453, 260715454, 143419106, -776307315, 552974634, -187252749, -367480864, -859467290, -838952399, 186293571, -661003285, 290808531, 57958365, 414203661, 592770096, 935936542, -483741899, -494067232, 665342187, -24733600, -851782115, 316963836, 928652996, -61227479, 633927691, -764314990, -236932119, -970850447, 986584827, -140901890, 901646787, -640670745, 946347069, 750088699, -746533752, -166721940, 223685665, 108925830, 339901588, -302714095, -757602829, 323130374, -873492891, -919981265, -380190612, 992899526, -836939849, 354664600, -609690691, -599861461, 982517365, -220416167, 884565755, -597860825, -230540368, -567323257, -416268461, -168826503, 561514596, -179711856, 315536394, 536086096, 111115714, -785676520, -889145225, 503999946, -806630876, -616282829, 627540091, -263443084, 670872806, 482241660, 791855345, -886929180, -437311230, 579402332, 312153069, -511994537, 136728314, -110476774, 908201328, 124865848, 2857333, 630884625, 967024670, -784333373, 740126397, -631392351, -277128893, -600308588, 749165538, 955214233, 549340427, 726369382, -194477203, 165754504, -521049540, 109084331, 570838205, -244594642, 597984235, 691733114, -13067911, 870475066, -339375214, -403051368, -296694373, -466889005, -419073649, -408726653, 225206490, 187585372, 670916664, 804804713, -860868324, -705491566, -885788550, 788902036, 239787186, -424584306, 469674913, 152157359, 69034995, -193884460, -105043709, 72807017, -245384924, 380045049, 649616129, -146230380, -124521393, 257550327, 909215813, 213244976, 194136578, 972653498, -807813957, -597438089, 808673329, -3213743, -997425383, -501022260, -825299917, 283744582, -216842456, -194597241, -559899116, 565625185, -481634742, 564890282, 263328991, 405146153, -102824834, -580576947, 222397888, -33542764, 631691628, -460352374, 250651291, 912602004, -102355501, -865508665, 980840826, 470403846, -241890373, -272378949, 47001248, -911002937, -183046528, -44709188, 135345183, -860764680, 620619962, 891659858, -763339758, -141934173, -167074468, -723880571, -243534000, 21883350, 802950568, -621769622, -534931633, -179407461, 802282315, -506248471, -657135370, 183248703, 120930147, -529708790, -104611432, 920926557, 605851544, -896937629, 903860505, -863881859, 273877368, -934329654, -617871964, -37315083, 8779480, -680390756, 334220782, -600878394, 166247076, -614421302, -730709352, -142036759, 177182230, -683747893, -649281856, -676028199, 144303967, -318617702, 93271138, 888832011, 620809583, -989483681, 663881472, 925824912, -387655746, -418337839, -895757399, -451080083, 514649968, -576289443, 502926520, 178387485, -75959760, 161438622, -386493309, -537124753, 56285315, -49949030, -198471894, 176715609, -875984462, 988167494, 433585453, 848420856, -588122894, 61938633, 524639038, -642074822, -720083750, 959849531, 730813732, 440972023, 314840927, -705438781, 877919934, -65907787, 558568779, 259881327, 668917225, 357242434, 766101652, -887315802, 125760116, 511388078, -855895652, -89647205, 620907615, 563811098, 572606651, 796241299, -618023309, -976175632, -123055094, 289350225, 901166623, -679632448, 742216560, -604913966, -51532458, 572074290, -403968436, 367307672, 322527647, 145338342, -183540342, 576244522, -6340193, -881165797, 7410453, 331361104, 75754037, -543403443, 422398876, -525784414, 389991012, 609112767, 567280235, -167307465, 984892757, -561327911, -443154308, -823671234, 116391266, 77482371, 219908, 21091249, 654971969, 142584287, 715079500, -379361199, -163367339, 155703977, -446302571, -257876689, -718956542, -111939677, -477058707, 15716805, -994813465, -960714466, 785218800, -319195689, 958638645, -284061553, 671965706, 768246187, 935032111, -543340390, 479749767, -387055937, -907228589, 394271220, 337824088, -427188484, 48236708, -215566177, -499901259, 213074987, 545161441, -232892564, 580734249, 874996370, 589950956, -212076735, -529697569, 126615468, -394887205, -694374521, -559962702, 606884995, -762992982, 479896249, -598027456, -845694734, 14195233, 945531891, -372357173, 878331302, 789229117, 310431527, -988203398, -755285472, 563359329, -216557696, 443094418, -172378019, 831253230, -970662423, 259514161, 671884385, -753303183, -685043474, 358610776, 722654601, 247889124, 960065217, 450996272, 875539034, -629306842, -377195667, 263086790, -550860438, -376741942, 811274647, -823540919, 660026509, 139294300, -935439301, 904715857, 950934364, -396088178, 229460467, -514191120, 495116191, 207933749, 930338505, 757330711, 562248541, -318759296, 382545645, -246765265, 774578537, -751062515, 616423006, -894602102, -121286813, -38964243, -883810362, -471330786, -150071268, -518216283, 216598238, -854836068, -444244352, -20299421, 573311170, -728376689, -355917280, -166742339, -955337589, -506430528, -969122187, -447807889, 82502114, 9911673, 363466260, 744351446, -350657925, 190700519, 279765080, 427267837, 159907167, -555539212, -711708064, -987035470, -188103071, 793348607, -636155305, -559117288, -917036179, -294795032, -532491721, -220567776, 462085133, 515649764, 15384615, 488064716, 302637753, 331631366, 982070392, 99642285, -430419845, -63892160, 41935450, -808597482, -758861951, 13482453, -5495137, -865390204, 743064986, 574311815, -391198519, 211597928, 431075693, -638282453, 510664973, -865807849, 345333193, 664574066, -579419844, -91817361, -184779617, 75346376, 125873610, 516140809, -391855629, -911471039, 625782529, -111813262, -942168998, 157781383, 585438527, -7490937, 975402738, -792695984, -960173983, 909975750, 851595538, -522927372, 358345600, -262826340, -625674682, -686079621, 947835311, -875216757, 771081440, 442624611, 719974791, -687117389, 848098371, -807166239, 312232580, 21258581, 313284614, -570487003, 433340340, 173436338, -823454098, -682238530, 417140342, 201205, 724031925, 805306941, 785241106, 799927702, 202372251, 255783472, -282202302, 886069398, -362799394, 503578273, 245205408, 952772058, -220944084, 556904832, 886405209, -117469947, -481304757, 70274528, -42622208, 966056960, 821497555, 883403483, -241852648, -712178543, -588310231, 334767541, 132724944, -129737409, 652460732, 848808184, -956765330, 47477982, 274528628, 508498716, 801767522, -95506017, 607844446, -777038045, -743326077, -622647726, -939761103, -345725437, 3352465, 232976743, 296912327, -60246864, 622915580, 342544779, 438161340, -670361118, 470048261, 435381002, 399064651, -41610959, 380808402, -999953811, -866739985, 302396261, 169807029, -416323947, 319645391, -750162640, 372499838, 972770579, 236990962, -315865169, -615382785, -193007525, 716245422, -741609642, 92893776, 780175725, -92239559, -987383836, -110184106, 55702126, 532209512, 310946378, -629449255, 550211179, 400936570, -16840902, -771729580, 183806267, 517633288, 112509911, -247466583, -130783148, -384030660, -370042597, 72052464, 103075060, -160802070, 481186063, 685620324, 958926847, -962432705, -434827410, 506530819, -118248396, -250299634, 879766585, -220173776, 686644558, -448836447, 658087845, -31993487, 248633142, 827323870, -626254498, 356981938, 371161740, 784274141, -383512275, -13332055, 829266375, -867022957, 828512655, -869972320, 80801804, 563314660, -864107, 591471894, -399189343, 873448293, -519725446, -135286449, -498996304, -275791387, 612180941, -124637763, -627816806, -793993992, 653565960, 512864478, -583080786, -151389052, -965501285, 182851325, -408511486, -693446470, 654512249, -899174250, 750073033, 442969290, -229844477, 234313467, 350137920, 249374118, 616272716, 986136333, -130778389, 126573394, 727018741, -770950417, 38759655, -821116855, -753677444, -160970451, -835624449, 320337516, 545945039, -857152318, 107014787, -682837537, -333168817, 689386099, -885332252, 325576996, -714056798, -205594159, -673964140, -588252259, -221376862, -35165687, -518196114, 63404852, 69593610, -796102474, -533203662, 789068288, -672776408, -906350729, 355433307, -222102748, 756235245, 31730307, -945861986, 851521317, -459789558, -725934096, 970819012, -110884731, 246160891, 972781110, -783756003, -999294359, -682480126, -758413361, -190066506, 41656368, 273225200, -908124403, -753038262, -191790073, 713725734, -74806629, -695283131, -169218848, -614472550, 190620875, -727750394, -83349137, 497265074, 433773108, -684984722, -891770569, 195667116, 268713517, 924758343, 740618165, -382245870, -693290034, -625168946, 100260206, -219677022, 993100233, 812962739, 857381969, 182414815, -249463662, -393844553, 226459151, -24801452, 643549454, -635740901, -449663938, -2934854, -663413942, 9359182, -884958262, -363466270, -711600974, -85825777, 499603687, 418714988, 455111552, -77635101, -482545225, 325287821, 433859182, 147613542, 439808234, -804641574, 596960356, -714573040, 877441182, 142069198, 65745208, -732667359, -323233819, -981329444, 974767662, -138871872, 291569610, 546294720, 231713714, -80225442, 589840267, -25393028, 769754358, 539689513, 532603320, 98813347, -6954131, 347606334, 762440868, 602588078, -494810996, 52008369, 868790099, -415067735, -200767253, -755871893, 399961812, 736750825, -951928500, 976495171, 399914957, 491870993, 89579210, 701609674, 537297709, -966269594, -807390782, 372176946, 59470598, 886633509, 141357705, -947620091, -599940343, 197505404, -874365737, 95473815, 34811214, -676459398, 292392010, 449017804, 24382622, -900330501, 828148840, 399841967, -714837474, 697327388, -444941616, 417659959, 699626876, 708178034, -115813988, 253248782, 211117613, -100438959, -704328963, -125452256, 180189816, 364950220, -83930544, -889045237, 316430385, 702569485, -337842247, -407721600, 451854569, -680501135, -512347412, 59593907, 857068966, 526556187, 121997853, 103830502, 787844696, -764361680, -98521930, 158447631, -602908690, 62758237, 489143304, -168497560, -565663842, 559125311, -847242353, 104040038, -317200835, -724875492, -645650397, -25604095, -263129832, -648724342, 839576123, 411525228, 446095801, -891206930, -864164807, 102983094, -119907005, -256761747, -626136194, 640658627, -7367735, 740747778, 97887775, -190452276, -779052387, -929043816, -472564973, 98063035, 184319038, -121689174, -31478469, 140761069, -426138203, -963693067, -225669674, -811355046, 643927744, -331386991, 325126654, -449312967, -40835730, -926614283, -687052157, -738023410, -505999656, -492237446, -113644347, 23979504, 532251436, 170812730, -665346123, -904986562, 212585349, 351238245, 462442562, -269216719, 574084698, -753634841, -705776011, -224710136, 188268567, -997060094, -52953058, 512823739, 690010681, 577529847, -621221305, -723780437, -556004618, -453641667, -728376219, 555460798, 353774293, -645372125, -601298109, 454275166, 8744735, 8141795, 887412927, 940035051, -312914408, 157508083, 534156160, 883702971, -659926650, -24026173, 872223590, -559387638, 627569176, 128308712, -145034557, -792916479, 798596817, 407257117, 556258220, 91543950, -360016999, 606447180, -459249736, -10328234, -292141380, 728900199, 510866106, 568477642, -25315999, 227806535, -75682181, -212068732, -813761508, -217455662, 468179961, -365180784, 205234737, -876437427, -742260615, -274744330, -58101594, -755444371, 880643580, 808783179, -693472281, 297269025, 296053767, 629718177, 679070941, 2539216, -944445859, 342938409, 59327880, 826955558, 810061515, 791370943, -26284595, -154613647, -394249451, 590996733, -454361806, -382409044, -19997680, -954609228, 209231059, 550591323, 971016385, 554317311, 617708727, -164111319, 908936326, -359619632, 760188503, 387301126, 214536361, 483066998, 477902506, 601321272, 530481383, 532523119, -696982184, -396164949, -429473520, 932834364, -606704635, -181941776, -313346182, 856753295, -435595219, 417286300, 560962503, -15541010, -940033276, 947395288, -763038180, -949524703, -522406108, -176440334, -201145306, -811951103, 390470422, -751609398, 939356545, -127946223, -309478415, 956422470, -960482104, 628305500, -522188592, -426607826, -797158919, -205303037, 94596551, 908317579, -664958943, -623135912, 485365625, 811732496, -144994753, -638879012, -68735034, -307369826, -947750849, -626935579, 522913758, 400440171, -313802815, 920457444, 771512769, 928931734, -910161390, -655453181, 700514193, 521282170, 685168304, 215219174, 452006309, -497733011, -774012867, 700488136, 586900095, 989021344, -560260745, 869393756, -728264229, 106043012, -68737653, 384284940, 276436488, -430732402, -566414137, 383586095, -253656592, 264729334, -408090918, 173437283, -113647549, -11410992, -121357515, 432768464, -457535726, 57308670, -486532090, 927559714, 95774978, -751271239, 819455692, -469559511, -182616525, -535271138, 59823234, -87895970, 945122332, 969042936, 747972332, 979171491, -670763132, -880202220, 150154023, 173665130, 689927721, -262559213, 958510825, 248654697, -161773661, 651597816, -706497349, -742170448, 21574291, -6913117, 47308194, -279218730, -285532771, 300575429, 537519317, 497508334, -699840490, 364583515, 668424648, 894250879, 742275818, -889719114, 810882420, -166910272, -658084132, -425566465, 769740523, 530157932, -173086971, -380134160, 211025041, -999893157, -542797056, 79851552, 27310792, 314216025, 166587020, -184341013, 270443596, -809195832, -711366084, 361361489, -292822886, -104107236, 811198649, -360522465, 118641868, 457568649, 32297284, -195254864, -289733799, 920040253, -760863818, 999679169, -914312663, -248864743, 262167147, 251362120, -603991644, 940810018, -722993169, -449539285, -879523786, -92246921, -259623398, -266232526, -269781388, 982405285, 383818309, -324552882, -466814728, -263654539, 983076149, -386998690, -651901563, -710321927, 355765741, 38775328, -974692739, -863211570, 205839892, -921312848, 962909339, 328039462, 757638536, -302541683, 985267587, -686151290, -69689993, 361791797, -546755229, 202029057, -539547992, 618386505, 345657609, 945861820, 870604000, -69042340, -905300484, 204815562, 538146625, -176943365, -174123697, 933097207, 893195944, -941437849, -292964236, -843106353, -436728058, 905008637, -547601024, 651240261, -603818952, -587728452, 744406655, -131130554, -795615192, 764777318, 544805112, -997374960, 2991011, -119677771, 509157452, 268102779, 677127263, -487980592, 67362612, -719366666, -616697011, -253640740, -524521081, -902205059, 528803475, 128998679, 657295099, -129333747, -596249019, 396331772, -746877908, 367663994, -53669591, 763884798, 247658355, -808236894, -34478511, -467543273, -511689694, 21047539, -961586632, -81737862, 755828582, -791729875, -177428765, -954929423, -638616832, 350585732, 993889237, 223118365, 69794045, -258742821, 599105746, -277539297, 408246600, -816873630, -583111061, -454922712, -570819200, 86993632, -532913885, 1164865, -925944254, -430749278, -999488575, -837291199, -815906128, 846157824, 838053317, 312018838, -80495610, -108450468, -413945018, -531745581, 94368850, -17523579, -21998371, 162673656, -958941777, 761132676, 882658858, 307365537, -848050794, 774746175, 840070573, 785553766, 125510350, 74117941, -993645249, 265433944, -995674280, 140083833, 733309755, 614115287, -891069576, 334642352, -98917008, -127176465, -367849677, 368047545, -492460671, -847073924, -72096258, -995525256, -975962647, 231923349, 112438660, -594109418, 329438681, -618208063, -97652460, 207684234, -720747214, 316133350, -876327985, 557586612, 441700778, -133648751, -584726637, -469330356, 986410233, -470872423, 35976095, 525633778, -978853234, -898993081, 144130082, 550990292, 356481958, 103333759, -464314604, -328821647, -522226253, 414764487, -656814205, -580033370, 209902844, -399431535, 352336686, -764110642, -281894414, 880182836, 803047612, -713799202, 701714084, -564727234, 44527332, 526317336, 254809169, -755010692, 437915516, -820655590, -892217040, 666926106, -302050895, 146405171, 344137458, 833122352, -390531799, -535967620, 605170474, -157645660, -21137778, -554146490, -519151443, 861355297, 245623233, -874445979, 219006632, 688322159, 126043545, -434913946, -899017414, -894211453, 45613988, 991587413, 320199946, -355091659, -166444252, 284800927, -597673186, -404789213, -79316504, 555677895, 757708175, 44765906, -370724466, 453018482, -993802138, 245036641, 885161500, 25794098, -927984297, -626394287, 319093824, 230936667, 333448051, 384718184, 297766283, -601592941, 426510157, -778391247, 438957394, 508655965, 50892027, 995803604, 923949628, -332642877, -286467234, -306221352, -78075748, 120466776, -993088224, 668177906, -365726275, -117911823, 609195936, -908696857, 838461630, 921505882, 991208721, -76084038, -113383815, -133013785, -787576933, 317343302, 837790130, 142458218, 81374734, 757861050, 87567210, -530295646, 129060582, 952180677, 621710797, -254300436, -869906083, -30455452, -367667057, -728167384, 536809591, -488966055, 555932874, 523751868, 947103615, 235669821, 685658747, 700922117, 85044694, -504115367, -607833166, 16894455, 604060138, 32645679, -225382483, 252115446, 942921866, -42092124, -778822863, -670785644, 316087952, 420111621, -861423415, 319494275, 894038112, -301460610, -229098897, -844782907, 279238474, 289740314, 337900693, -796379667, -989908610, 188509154, 717567061, 763045451, 425278866, -631339267, 65537281, 977382142, 605242791, -670240747, 862749086, 660177628, -136815320, -495764981, 93730822, -607763840, -91079928, 312779786, -989383251, 26334420, -858517409, -213493949, 770428979, -211855066, 528470722, 146303405, -365344093, -444758138, 921479789, 641372083, 767451947, 455526816, -658928385, -404220474, 885634184, 135955854, -708950262, 148785600, -293220825, -977328426, 35315932, 942026799, -217606519, -432108878, 661299680, -252382117, -901344948, 393894487, -466867606, 240660537, -462430353, 240199969, -372944848, -157131535, -918139228, 707015071, 609772696, 460628472, -529960695, -860620909, -973060113, -32473291, -253456236, -987912502, 623467415, -723663701, -92548803, 87734084, -765574061, -747334053, -87081455, -461293151, 93311365, -815291390, -454929241, 879887918, -946179179, 357064729, 568650690, 343080044, -99789696, -690366762, -978811162, -956901966, 824091820, 289392670, 557264571, -13871797, 536213802, 858341204, -171704826, -465413034, -979191244, 597209042, -673147690, 284366587, -197734345, 685513647, 823529320, 786305889, -107039640, -285150517, -464636804, -597807303, -730875571, -729895665, 555698388, -586845303, 325440276, 98904079, 254227111, -461130626, 355258797, 947467278, -810178512, -677272378, -89361026, -684007308, 347110427, 227590692, 803478932, 228215007, -75119336, 707587263, -277087453, 583902058, 761246218, -462945537, 883952539, -290632788, 311337555, 556847197, 532594298, 645494956, 902471461, 70523429, -128252533, 711430214, -478177688, 722503423, 118841145, 534369846, 603957085, 946143566, -555162127, 378116229, -243818647, -302796554, -937031364, 4141899, 839346051, 548120059, 15116292, 520087197, 233492600, 184405937, 657993566, -386681132, 732767609, -231542147, 37810754, 805226611, 510141309, -873683651, 355610558, 36056305, -856660160, 203156102, -594959484, 404330784, 767701123, -779099154, -669724050, -104311236, -963542315, -662127288, -887835061, 885834256, -394789660, 32783896, -650379668, 632308392, -853836721, -862090002, 698013729, -522149399, -79328972, -888669805, -996287131, -646304985, 108702528, 249107060, -950339973, -213212542, 810324742, -400724048, 493891563, 472869542, 62891521, -910958256, -461363153, -138907283, 533476942, 868589500, 292932918, 466257116, -525731061, -361358222, 141980493, -344040756, 274541144, 728872749, -114250767, 202991958, -320859324, -492860227, 872498590, -178427442, -145301995, 611678321, 94636877, 902976672, -196029056, -332654077, -256687524, -689646276, 197626356, -236982461, 652244729, 88988714, 185358461, -360894727, 469991712, -513258257, 282392305, 107738700, 631848938, -623551064, 436571504, -97803295, -956158972, -475727592, 505148007, -843666358, 279596238, -131189609, -308108144, -574875109, 727143011, -597771997, 677318761, -703171388, 756556959, 831539823, 70881998, 474876256, 613355875, -377058679, -690112676, 801487628, -523292134, 401051028, 738402525, 874942125, -554619533, 929442293, 495557415, 253003703, 105350561, -645173436, -24665264, -911587971, -790033709, 724021795, 876101532, -650145583, 431350644, -159310746, -285736088, -638809190, 989211027, -782584631, 566194737, 129133251, 122883695, -827353979, -236002225, 928077298, -540958876, -478748089, -903781758, -853756642, 862583093, -951597876, -90481864, -973278082, 738114879, -650765773, 597406090, 225859022, -879568559, 276739184, 782144148, 345346208, 225714127, -20497262, -990030103, 340091715, -986020576, 612429468, -977949036, 24767269, -517197241, 125436511, 8686945, -756440428, 355389538, 140792078, 813732822, 706815716, -386529003, -990941074, -186083255, 115176821, 853115466, 698881123, -665060581, 385183684, 725287464, -246363552, 909390410, 713265504, 733892488, -527654709, -467305358, 794747640, 232462387, 165177684, 32554446, 638492749, 135381474, -29428623, 35420411, 612453257, -157460239, -171233579, -653686166, 860289590, -481267125, -690223165, -915749966, 745549102, -768207879, -261977399, -383315408, -470178595, -569706481, -933961023, 989801897, 694736209, 108312569, -528422608, 189820878, -869446511, -982022729, -460941728, -23490611, -104961277, -858834441, 965236450, 83777697, -725432661, -342698490, 491730717, 230679568, 375294207, -218040282, 809723868, 677915898, 100649169, -787799151, -61575156, 240062881, -49493936, 614806407, 314634673, -993853691, 371327092, 674202794, -776311503, 255140885, 277010845, -583244480, 264599519, 412430616, 712511425, 336554623, 894568223, 360474484, 801977142, 709910748, -847323210, 185295523, -377508053, 405783610, 597816717, -16291796, -828068458, 800211051, -317322606, 224596183, -988402712, -509705101, 929114904, 988045779, 382920916, -114204661, 716354023, -477631309, -466038421, -198310364, -741816226, 579706736, -357704328, -935554580, 744072050, 504343900, -466384868, -379957537, 629811984, -69176089, -451547128, -290405049, 266708172, 522316838, -128726841, -130728937, 697465414, 491072531, -460209247, -174005102, -600579795, 410317762, -721182758, 58767135, -626652760, -180758922, -715716224, 265600992, 51518859, 425436840, -106960287, 355768718, 532652987, 277809198, 56748553, -676413433, -248390521, -163571643, -429298767, -871041925, -295795300, 137781141, -878031138, 875701368, -153758881, -265282085, -614855979, 814609937, -173908913, -836613754, 951577329, 456023308, -398919356, -793646344, -776889906, 965257290, 734657542, 341553668, 87460608, -380654967, -796584304, -75194798, -857503739, -728747921, 312233672, 148770736, -587501197, -860216929, -475809893, -42120848, 745990572, -706044125, -40482686, -291626097, -295195454, 317808484, -714178179, 997053063, 321079256, -744803777, 638216560, -423634724, 450947343, -18671526, 41021597, 83980518, 622092634, -869507249, 91696643, 717343855, -817454549, -767109964, 620310403, -130393245, -36702681, 218195026, -972642050, 661175455, -774899708, 248151968, 959233465, -530013137, 622744058, 505523625, -502736869, -959702050, -346064088, -544956571, -860939453, 857621429, -16828981, 285402588, 469260391, 636021435, -625517981, 536309790, 226269923, 191196405, -736497874, -159873681, 477921303, 623909545, 97558326, -381520763, -529498846, -944616463, 253722988, -208654094, 559564438, 731963161, -355216920, 938764914, -834901363, -728190070, 994104632, 564227844, -52183810, 67773809, 813521791, -766281024, 12395728, -755896055, 985175761, 442360452, -997330740, 710568206, 833187451, 246710899, 15922992, 762057564, 573982464, -618982335, -644905669, -796967626, 110414143, -960217265, -323599690, -403050971, 170845096, 271855756, 597125971, 758621369, -985132496, -589471759, -724589982, -779227217, 42925375, -241527677, -534489690, 955201459, 82273282, -451490667, -470357558, -91332201, 536399164, -23575090, 18679890, 978316982, 519150811, 843163438, 57562159, -722455092, 975099151, 427251707, -141193643, -414151446, -651080868, -413278973, 773005468, -850491121, -984289027, -643695874, 330087089, 889664197, -712239856, 248831962, 849301943, -177587007, 976983058, 845523771, -454589814, -717991119, -679685951, -200414813, 288187551, -623544898, 465694499, 685303115, -69209062, -395792448, 76343779, 736999170, -545407609, 346298600, -676028948, -997569787, 720615933, 260007285, -395704187, 183085992, 489956486, -500531855, -900617850, 381604780, -177392759, -131414689, -411520593, -181005873, 251184263, -489576523, 106142809, 716146857, -111086647, 64311224, 502737576, -927334279, -277220817, 689163776, -558060739, -778656629, -222553836, 438053361, -685243298, -491849872, -597268216, 72561347, -781930970, 555330293, -313177017, 213334995, 628736190, 631392194, 514248868, -25044727, 213407700, 133831677, 645823491, 589164714, -255224905, -52924916, 593487474, 387874958, 55141290, 264193643, -927711712, 614695699, -473135352, -607795549, -607567311, -320767647, 744829231, 421847759, 544335579, 418161640, -222986, -7068343, 114984860, -885161576, -325453047, -32199982, 533669607, -276744844, -522528835, -385457949, 297835032, -135120817, 691085840, -880990488, 824659170, 19713745, 924576445, 59920474, 169794415, 269514288, -185222092, 447680416, 912776246, 947816297, -254447504, 986977694, -71054444, 951951880, -549854713, 8357358, -913472193, 654283124, 688749887, 735224517, -705505019, -91688901, -317938297, -648194973, -701595029, -654523694, 83171542, -194977636, -903793473, 997994785, -893143363, -208706374, 74801924, 703525411, -635925911, -416486888, -398488199, 698725143, -203870916, 582740595, -155686824, -494568783, 925544728, 257565645, 585090724, 918275353, -588822841, -845979049, 819108075, 713472255, 827640882, -448046496, 49421574, -474348511, 67310272, 85904796, -18754634, 448138250, -515202521, 411496077, 226467382, -54584711, -922274980, 89244562, 130907398, 844814014, 55443658, -902871517, 863283461, -892465610, 569072020, -799593901, 827864226, -352598722, 207759254, -92057955, -127634902, 351092258, -442370474, -246366631, -819519667, 956887107, -152412860, -329076043, 313197384, 795284363, -709717865, 134619216, 851757476, -256553652, 794040931, -425980154, -751164529, -398074632, -589457920, 737903985, 726490431, 752649544, 557197550, -946188614, 629092185, 106769713, -513779343, 951456596, -762133803, 945127160, 987368879, 658836137, 580489541, -903886308, 909228013, 396524794, -761073717, 277507519, -714437307, -776708809, -693089197, 471128672, 367406941, 711107755, -819524924, -29138393, -3134513, -621632875, -937626010, -248133743, -574088421, -962440429, 749745195, -80638955, -956481677, 796004788, 147497321, 180439499, 813904068, -294534236, -754170570, 637619512, -633172719, 527218107, -631353831, -735985003, 482301625, -970521320, -488320884, 775416295, 174621783, -726086667, -971384358, 892247020, 985832007, 30417640, -102054977, -848455749, -14594566, -195650359, -130360019, -878064410, -813885862, 826633032, 46629741, -13086799, 321882178, -986961424, -760501728, -906894703, 862925520, -483004719, -756731818, 456732723, -39894547, 559605706, 355605858, -529702470, -248373804, -501359233, 44809239, -937983375, -861815148, -358927346, 410831995, -955735967, 600756913, 777870333, 566234519, 398870874, 779432982, -531091376, -845973842, -608100610, 944086363, -809617379, -716407061, -638360081, -252099649, 945594989, 193054749, 204656944, 766711309, 531096705, 126928889, -160149272, 593846225, -794344357, -807927158, 978455274, -206756774, -684458503, 655981363, 25950435, -173971768, 570289721, -398056574, 828165370, 49313703, -103103858, 987695395, 310549223, -110864173, -715759689, -209438883, -592758983, -173838838, -46822092, 822339395, 804496775, -89868060, -987610403, -102504012, -691738246, -150129147, -36810342, -462549937, 153775722, 203242977, -475492657, 6533797, -744055123, 425795776, -532185893, -314870598, -502695399, 292006054, 345892537, -35759816, 807412399, 121493137, 116747263, -926323955, 259257483, -700796775, -373843073, -139062490, 260057423, 920027367, -14737920, -204735261, -222841829, -373626516, 498027835, 235283073, 853407142, -347432140, -563747901, 893263415, 946852805, -553699316, 230726352, -342872263, 484779034, 231783456, -282264067, -339004790, -328053304, -291997266, -221857799, -556024012, -597063441, -109061154, -748188248, -183079122, -225197305, 994765990, -381116308, -122557046, -857062378, -821441070, 15971148, -554840472, 906183528, -310987352, -942583382, -707884072, 769064246, -881816940, 898747535, -206602277, -468166182, -321543406, -832983257, 43380198, 949334774, 11385765, -376905485, 644186625, 587201097, -717640520, -959412177, 624540895, -223320092, -267153703, 687246272, 317851188, 812906293, -597749041, 884295145, 478584047, 670580245, -28773866, 381605311, -858383090, 874487022, -294966489, 611723884, 808246524, -960036524, 95924254, 557671399, 351868673, -583465326, 375469879, -306523056, -368343072, 881061462, 545970118, 706509799, 362492413, -903871225, -858370062, 757618903, 622380196, -566445645, -326849985, 467799617, 242816896, 783595061, 99377470, -546409009, 503815091, 262702622, 699243303, 277461568, -578371123, -4679819, -669474503, -174488063, -404340715, -963040521, -988303444, 876968199, 785952452, 420772001, 307471946, 208780107, -130753996, -596075106, -289399487, -298258158, 659395242, -756424588, -674127934, 721015253, 671324954, -245141250, 882430840, -591615352, 962627910, -917743928, 569854895, 698068759, 207129927, -155377949, 549139323, -948388045, -594187283, -850500956, -623131038, 165768789, -561839522, 301376230, -829104467, 406991350, 913188197, 240094845, 430914507, 782210393, -554053399, -187229428, 561206973, 274479402, 538928594, -421436662, -580082645, 496635029, -494622723, -200666105, 685051358, 516839056, -761442771, 442877251, 64597312, -825147876, -741325066, -958340891, 51786353, -852691956, -729420126, -704249514, -449373831, -961877818, 236481473, 787400743, 816448022, 272839393, 470960499, -372152621, 996310954, 550823858, 567429986, -458939353, -434984329, 534001075, -882532446, -934050802, 516224359, -922116270, -893264738, -616647808, 194456900, 452049576, 467610538, 140981576, -128338092, 688773738, -323786882, -471413033, 241953888, -419030191, 313673400, 786096372, 819490834, 794516133, 422324404, -651601988, -802903890, -490056329, -739079955, 103635246, -535301773, 660984777, 62592062, -424088382, 700664625, 850087564, -962274422, 202725208, -562434387, -771727360, -598040651, 404602199, -316547499, -923907356, 368004059, -397938865, 554456673, -807254600, -861298029, 833841210, 463341505, 822984467, 693964989, 8160549, -494223091, 506180793, -575881298, 669795974, -52606136, 382419801, -825490051, 249437062, -783425481, -37820854, 125900681, -235730997, -721490177, -47831242, 59627575, 522767239, -289984267, -196042148, -108880743, 507280517, 143871888, 194027415, 767968453, -167806760, 80417366, -153843640, -216917387, 552926594, -181339024, 473042261, 693404980, 182744013, -65287130, -817397241, 596715278, 166499808, 244240536, -16916773, -82406657, 556419192, -239820210, 265824626, -40005925, -770578023, 41218978, -231308629, -337668192, 578001101, 244408591, -88023459, 72898630, -627396505, 309956542, 5727810, 181801845, 895209297, 739139924, -946247659, 713593908, -72751501, 695373197, -309444720, 77745444, 914890891, 266620027, -926215390, 482400003, 826149664, 688973591, -635431967, 344534897, 249384383, 638289786, 195217244, -145375131, 507624537, 130403999, 402257762, 366836809, -527648211, 696816235, -720121660, -703706585, 903683861, -900082730, 137747832, 863193002, -505697638, -281728840, 954237799, -849790695, -820403458, 796670238, -867856242, 826665885, 717556211, -621459229, -173397239, -873807628, 159465500, -733563003, -662058411, 464779721, 344947222, -987550565, 260083197, 357715444], "dtype": "int32"}], "e": 1869}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(argmin, json.loads(_TESTS), 'cuda', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")